<!-- # C2 回路による量子乱数生成 -->

# Quantum Random Number Generation with the C2 Circuit

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- 5 個の量子ビットのそれぞれに C1 回路を並べて作用させる **C2 回路**を Qiskit で実装し、得られたビット列の偏りとエントロピーを調べます。1 回の測定（1 shot）で 5 ビットの乱数が得られます。流れは次のとおりです。 -->

We implement the **C2 circuit**, which applies the C1 circuit in parallel to each of 5 qubits, in Qiskit and examine the bias and entropy of the resulting bit string. Each measurement (1 shot) yields 5 random bits. The workflow is as follows.

<!--
1. C2 回路を作る（重ね合わせを作るゲートを Hadamard ゲート $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ の中から 1 つ選ぶ）
2. 回路を量子コンピュータのネイティブゲートに変換（transpile）する
3. **ノイズなし**のシミュレータで測定し、3 つの見方（量子ビットごと・5 ビットの同時分布・連結したビット列）で評価する
4. 実機相当の**ノイズあり**のシミュレータで測定し、同じように評価する
5. 同じ測定を 10 回繰り返し、結果のばらつきを見る
6. ノイズの強さを段階的に変え、結果への影響を見る（5.9）
-->

1. Build the C2 circuit (choose one gate to create the superposition from the Hadamard gate $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, and $R_Y(\pi/2)$)
2. Convert (transpile) the circuit into the native gates of the quantum computer
3. Measure on a **noiseless** simulator and evaluate from three viewpoints (per qubit, the 5-bit joint distribution, and the concatenated bit string)
4. Measure on a **noisy** simulator that mimics real hardware and evaluate in the same way
5. Repeat the same measurement 10 times to see how much the results vary
6. Change the noise strength step by step to see its effect on the results (5.9)

<!-- このノートブックは、C1 回路のノートブック（`C1_circuit_textbook.ipynb`）を先に読んでいることを前提にしています。C1 と共通の内容（ゲートの行列、ノイズの式、$\lambda$ の決め方など）は要点だけを書くので、詳しくは C1 のノートブックの説明パートを参照してください。 -->

This notebook assumes that you have already read the C1 circuit notebook (`C1_circuit_textbook.ipynb`). Material shared with C1 (gate matrices, noise formulas, how $\lambda$ is chosen, and so on) is only summarized here, so see the explanatory part of the C1 notebook for details.

<!-- この説明パート（A〜G）で理論と使う道具をひととおり説明し、後半の各節（1.〜5.）でその内容をコードと並べて一つずつ確かめていきます。 -->

This explanatory part (A–G) goes through the theory and the tools used, and the later sections (1.–5.) then check that material one step at a time alongside the code.

---

<!-- ### A. 先行研究と C2 回路 -->

### A. Prior work and the C2 circuit

<!-- このノートブックの回路と、ノイズのパラメータの数値は、次の論文にもとづいています。ノイズモデルの組み立て方は論文のものではなく、このノートブックで選んだものです（E を参照）。 -->

The circuit and the numerical noise parameters in this notebook are based on the following paper. How the noise model is assembled is not taken from the paper; it was chosen for this notebook (see E).

> A. Gnatowski *et al.*, "True Random Number Generators on IQM Spark," arXiv:2512.09862 (2025).

<!-- C2 回路は論文で次のように説明されています。 -->

The paper describes the C2 circuit as follows.

> "Circuit C2 is a very popular and natural extension of C1 and works by applying C1 circuit to $q > 1$ qubits."

<!-- つまり、C1 回路（$|0\rangle$ に 1 個のゲートを作用させて測定する）を、複数の量子ビットで同時に行う回路です。このノートブックでは論文と同じく、Odra 5 の全量子ビットにあたる $q = 5$ 個を使います。 -->

In other words, it is a circuit that runs the C1 circuit (apply one gate to $|0\rangle$ and measure) on multiple qubits simultaneously. As in the paper, this notebook uses $q = 5$ qubits, which is all the qubits of Odra 5.

<!--
```
q0: |0⟩ ──[ * ]──[M]──
q1: |0⟩ ──[ * ]──[M]──
q2: |0⟩ ──[ * ]──[M]──   （* は H_Gate, Rx(π/2), Ry(π/2) のいずれか。5 個とも同じゲート）
q3: |0⟩ ──[ * ]──[M]──
q4: |0⟩ ──[ * ]──[M]──
```
-->

```
q0: |0⟩ ──[ * ]──[M]──
q1: |0⟩ ──[ * ]──[M]──
q2: |0⟩ ──[ * ]──[M]──   (* is one of H_Gate, Rx(π/2), Ry(π/2); the same gate on all 5 qubits)
q3: |0⟩ ──[ * ]──[M]──
q4: |0⟩ ──[ * ]──[M]──
```

<!-- C1 のノートブックと同じく、Shannon エントロピー $H_{\mathrm{Shannon}}$ と区別するため、Hadamard ゲートを $H_{\mathrm{Gate}}$ と書きます。コード内のゲート名やグラフの凡例は `"H"` のままです。 -->

As in the C1 notebook, the Hadamard gate is written $H_{\mathrm{Gate}}$ to distinguish it from the Shannon entropy $H_{\mathrm{Shannon}}$. Gate names in the code and in plot legends remain `"H"`.

<!--
論文が挙げる C2 回路の特徴は次のとおりです。
- **長所**：1 shot で $q$ ビット得られるので、生成速度が C1 の $q$ 倍になる。量子コンピュータを効率よく使える。
- **短所**：同時に使う量子ビットが増えるので、量子ビット間の干渉（crosstalk）が増え、出力の質が下がるおそれがある。また、校正が不十分で読み出し誤差の大きい量子ビットを使ってしまう可能性も高くなる。
- ただし良い量子ビットを使う可能性も同時に高くなるので、論文は「一部の量子ビットが他を補い合い、結果はむしろ安定しやすい」と予想しています。
-->

The paper lists the following characteristics of the C2 circuit.
- **Advantages**: Each shot yields $q$ bits, so the generation rate is $q$ times that of C1. The quantum computer is used efficiently.
- **Disadvantages**: More qubits are used simultaneously, so interference between qubits (crosstalk) increases and the output quality may degrade. It also becomes more likely that poorly calibrated qubits with large readout errors are used.
- However, the likelihood of using good qubits increases at the same time, so the paper anticipates that "some qubits compensate for others, and the results tend rather to be stable."

<!-- なお論文によると、$H_{\mathrm{Gate}}$ を使う C2 回路は文献でよく使われていますが、各量子ビットに $R_X$ や $R_Y$ を 1 個だけ使う C2 回路を用いた先行研究はありません。 -->

According to the paper, C2 circuits using $H_{\mathrm{Gate}}$ are common in the literature, but there is no prior work using a C2 circuit with a single $R_X$ or $R_Y$ on each qubit.

---

<!-- ### B. 量子乱数生成の原理 -->

### B. Principle of quantum random number generation

<!-- 各量子ビットには C1 回路と同じゲートが独立に作用するので、測定直前の 5 量子ビットの状態は、1 量子ビットの状態の**テンソル積（直積状態）**になります。$H_{\mathrm{Gate}}$ または $R_Y(\pi/2)$ の場合は -->

Since the same gate as in the C1 circuit acts independently on each qubit, the 5-qubit state just before measurement is a **tensor product (product state)** of 1-qubit states. For $H_{\mathrm{Gate}}$ or $R_Y(\pi/2)$, it is

$$
|\psi\rangle = \left(\frac{|0\rangle + |1\rangle}{\sqrt{2}}\right)^{\otimes 5}
= \frac{|00000\rangle + |00001\rangle + \cdots + |11111\rangle}{\sqrt{32}}
$$

<!-- です（論文の式 (2)）。$R_X(\pi/2)$ の場合は各量子ビットが $(|0\rangle - i|1\rangle)/\sqrt{2}$ になり、係数に $-i$ の累乗がかかりますが、絶対値はどれも $1/\sqrt{32}$ です。したがってボルンの規則により、5 ビットの測定結果 $x \in \{0,1\}^5$ はどれも -->

(Eq. (2) in the paper). For $R_X(\pi/2)$, each qubit becomes $(|0\rangle - i|1\rangle)/\sqrt{2}$, and the coefficients acquire powers of $-i$, but every magnitude is $1/\sqrt{32}$. Therefore, by the Born rule, every 5-bit measurement outcome $x \in \{0,1\}^5$ occurs with probability

$$
P(x) = \left|\frac{1}{\sqrt{32}}\right|^2 = \frac{1}{32}
$$

<!-- の確率で現れます。これは「公平なコインを 5 枚、独立に投げた結果」と同じです。 -->

This is the same as "the result of independently tossing 5 fair coins."

<!-- **Qiskit のビットの並び順**：Qiskit の測定結果の文字列（`get_counts()` のキーや `get_memory()` の要素）は、**右端が量子ビット 0**、左端が量子ビット 4 です。たとえば `'00010'` は「量子ビット 1 だけが 1、ほかは 0」を表します。このノートブックで量子ビット $q$ の値を取り出すときは、文字列の右から $q$ 番目（0 始まり）を読みます。 -->

**Qiskit's bit ordering**: In Qiskit's measurement-result strings (the keys of `get_counts()` and the elements of `get_memory()`), the **rightmost character is qubit 0** and the leftmost is qubit 4. For example, `'00010'` means "only qubit 1 is 1 and the others are 0." In this notebook, the value of qubit $q$ is read as the $q$-th character from the right (counting from 0).

---

<!-- ### C. 評価指標 -->

### C. Evaluation metrics

<!-- C2 回路の測定結果は 1 shot あたり 5 ビットなので、次の 3 つの見方で評価します。$N$ を shot 数とします。 -->

Since the C2 circuit yields 5 bits per shot, we evaluate it from the following three viewpoints. Let $N$ be the number of shots.

<!-- **(1) 量子ビットごとの評価**：量子ビット $q$ だけを取り出し、C1 と同じ指標を計算します。量子ビット $q$ で 1 が出た回数を $n_{q,1}$ として -->

**(1) Per-qubit evaluation**: Take only qubit $q$ and compute the same metrics as for C1. With $n_{q,1}$ the number of times 1 occurred on qubit $q$,

$$
\hat{P}_q(1) = \frac{n_{q,1}}{N}, \qquad \mathrm{bias}_q = \left| \hat{P}_q(1) - \frac{1}{2} \right|
$$

<!-- です。統計的な揺らぎの目安は C1 と同じく $\mathbb{E}[\mathrm{bias}] \approx \sqrt{2/\pi}\,/\,(2\sqrt{N})$ です。 -->

As in C1, the reference value for statistical fluctuation is $\mathbb{E}[\mathrm{bias}] \approx \sqrt{2/\pi}\,/\,(2\sqrt{N})$.

<!-- **量子ビット間の相関**：理想的には 5 個の量子ビットは互いに独立です。量子ビット $i$ と $j$ の値の相関係数（Pearson）を -->

**Correlation between qubits**: Ideally the 5 qubits are mutually independent. The (Pearson) correlation coefficient between the values of qubits $i$ and $j$ is computed as

$$
r_{ij} = \frac{\hat{P}(b_i = 1, b_j = 1) - \hat{P}_i(1)\,\hat{P}_j(1)}{\sqrt{\hat{P}_i(1)\bigl(1-\hat{P}_i(1)\bigr)\,\hat{P}_j(1)\bigl(1-\hat{P}_j(1)\bigr)}}
$$

<!-- で求めます。独立なら $r_{ij}$ はおよそ $\pm 1/\sqrt{N}$ の範囲で揺らぐだけです（$N = 2 \times 10^5$ で約 $0.0022$）。 -->

If they are independent, $r_{ij}$ merely fluctuates within roughly $\pm 1/\sqrt{N}$ (about $0.0022$ for $N = 2 \times 10^5$).

<!-- **(2) 5 ビットの同時分布**：32 通りの結果 $x$ の頻度 $\hat{P}(x) = n_x / N$ を見ます。理想値 $1/32$ からのずれ $\bigl|\hat{P}(x) - 1/32\bigr|$ と、分布全体のエントロピー -->

**(2) 5-bit joint distribution**: We look at the frequencies $\hat{P}(x) = n_x / N$ of the 32 outcomes $x$. We compute the deviation $\bigl|\hat{P}(x) - 1/32\bigr|$ from the ideal value $1/32$ and the entropies of the whole distribution,

$$
H_{\mathrm{Shannon}} = -\sum_{x} \hat{P}(x) \log_2 \hat{P}(x), \qquad
H_{\min} = -\log_2 \max_{x} \hat{P}(x)
$$

<!-- を計算します。どちらも最大値は 5 ビット（1 ビットあたり 1）です。 -->

The maximum of each is 5 bits (1 per bit).

<!-- ここで注意が必要です。32 個の頻度はそれぞれ標準偏差 $\sigma = \sqrt{(1/32)(31/32)/N}$ で揺らぐので、**その最大値は $1/32$ より系統的に大きくなります**（32 個の標準正規分布の最大値の期待値は約 2.07 なので、$\max_x \hat{P}(x) \approx 1/32 + 2.07\,\sigma$）。そのため、理想的な回路でも $H_{\min}$ は 5 より小さく見えます。$N = 2 \times 10^5$ では約 4.963 ビット（1 ビットあたり約 0.993）です。このノートブックでは、この目安をシミュレーション（多項分布からの抽出）で求めて比べます。 -->

A caution is needed here. Each of the 32 frequencies fluctuates with standard deviation $\sigma = \sqrt{(1/32)(31/32)/N}$, so **their maximum is systematically larger than $1/32$** (the expected maximum of 32 standard normal variables is about 2.07, so $\max_x \hat{P}(x) \approx 1/32 + 2.07\,\sigma$). As a result, $H_{\min}$ appears smaller than 5 even for an ideal circuit. For $N = 2 \times 10^5$ it is about 4.963 bits (about 0.993 per bit). In this notebook, this reference value is obtained by simulation (sampling from a multinomial distribution) and used for comparison.

<!-- **(3) 連結したビット列（flatten）**：各 shot の 5 文字を、Qiskit の文字列の順（量子ビット 4, 3, 2, 1, 0 の順）のまま shot 順につなげ、長さ $5N$ のビット列にします。これは論文の Table III の "flatten" の行や、後の SP 800-90B・抽出器のノートブックに渡すビット列にあたります。このビット列に C1 と同じ指標（$\hat{P}(1)$, bias, $H_{\mathrm{Shannon}}$, $H_{\min}$）を計算します。ビット数が $5N$ なので、揺らぎの目安は $\sqrt{2/\pi}\,/\,(2\sqrt{5N})$ です。 -->

**(3) Concatenated bit string (flatten)**: The 5 characters of each shot are joined in shot order, keeping Qiskit's string order (qubits 4, 3, 2, 1, 0), to form a bit string of length $5N$. This corresponds to the "flatten" row of Table III in the paper and to the bit string passed to the later SP 800-90B and extractor notebooks. The same metrics as for C1 ($\hat{P}(1)$, bias, $H_{\mathrm{Shannon}}$, $H_{\min}$) are computed for this bit string. Since the number of bits is $5N$, the reference value for the fluctuation is $\sqrt{2/\pi}\,/\,(2\sqrt{5N})$.

<!-- > **注意**：(3) の min-entropy は各ビットが i.i.d. だと仮定した値です。量子ビット間に相関があると、(3) だけでは見逃します。(1) の相関係数と (2) の同時分布は、その仮定を確かめるためのものです。相関などを考慮した厳密な評価（NIST SP 800-90B）は別のノートブックで扱います。 -->

> **Note**: The min-entropy in (3) assumes that each bit is i.i.d. Correlations between qubits would be missed by (3) alone. The correlation coefficients in (1) and the joint distribution in (2) are there to check that assumption. A rigorous assessment that accounts for correlations and similar effects (NIST SP 800-90B) is covered in a separate notebook.

---

<!-- ### D. ネイティブゲートと transpile -->

### D. Native gates and transpilation

<!-- IQM Spark（Odra 5）のネイティブゲートは $R_X$, $R_Y$（1 量子ビット）と $\mathrm{CZ}$（2 量子ビット）です（論文の Table II）。C1 と同じく、Qiskit の transpile では $H_{\mathrm{Gate}}$ は -->

The native gates of IQM Spark (Odra 5) are $R_X$, $R_Y$ (1-qubit) and $\mathrm{CZ}$ (2-qubit) (Table II of the paper). As in C1, Qiskit's transpiler decomposes $H_{\mathrm{Gate}}$ into the two gates

$$
H_{\mathrm{Gate}} = e^{i\pi/2}\, R_X(\pi)\, R_Y(\pi/2)
$$

<!-- の 2 個のゲートに分解されます。C2 回路ではこれが 5 個の量子ビットそれぞれに起こるので、ノイズを受けるゲートは $H_{\mathrm{Gate}}$ で 10 個、$R_X(\pi/2)$・$R_Y(\pi/2)$ で 5 個です。C2 回路は 2 量子ビットゲートを使いませんが、C1・C3 と設定をそろえるため、基底ゲートは `["rx", "ry", "cz"]` にします。 -->

In the C2 circuit this happens on each of the 5 qubits, so the number of gates receiving noise is 10 for $H_{\mathrm{Gate}}$ and 5 for $R_X(\pi/2)$ and $R_Y(\pi/2)$. The C2 circuit uses no 2-qubit gates, but the basis gates are set to `["rx", "ry", "cz"]` to keep the settings consistent with C1 and C3.

---

<!-- ### E. ノイズモデル -->

### E. Noise model

<!-- C1 のノートブックと同じノイズモデルを、5 個の量子ビットすべてに**同じ強さで**加えます。数値の根拠と $\lambda$ の決め方は C1 のノートブックの説明パート E を参照してください。 -->

The same noise model as in the C1 notebook is applied to all 5 qubits **with the same strength**. See Explanation Part E of the C1 notebook for the basis of the numbers and how $\lambda$ is chosen.

<!--
| パラメータ | 論文の値 | このノートブックでの設定 |
|---|---|---|
| 1 量子ビットゲートの忠実度 | $\ge 99.9\%$ | 脱分極パラメータ $\lambda = 0.00198$（熱緩和と合わせた平均ゲート忠実度が 99.9% になる値） |
| 読み出しの忠実度 | $\ge 97\%$ | 読み出し誤差 $\varepsilon = 0.03$（対称） |
| 1 量子ビットゲートの時間 | $\le 20$ ns | $t_g = 20$ ns |
| 緩和時間 $T_1$ | 0.964 ms | $T_1 = 0.964 \times 10^6$ ns |
| 位相緩和時間 $T_2$ | 1.155 ms | $T_2 = 1.155 \times 10^6$ ns |
-->

| Parameter | Value in the paper | Setting in this notebook |
|---|---|---|
| 1-qubit gate fidelity | $\ge 99.9\%$ | Depolarizing parameter $\lambda = 0.00198$ (the value that gives an average gate fidelity of 99.9% including thermal relaxation) |
| Readout fidelity | $\ge 97\%$ | Readout error $\varepsilon = 0.03$ (symmetric) |
| 1-qubit gate time | $\le 20$ ns | $t_g = 20$ ns |
| Relaxation time $T_1$ | 0.964 ms | $T_1 = 0.964 \times 10^6$ ns |
| Dephasing time $T_2$ | 1.155 ms | $T_2 = 1.155 \times 10^6$ ns |

<!-- 加えるノイズは (a) 脱分極誤差、(b) 熱緩和誤差（(a) と (b) を合成して $R_X$, $R_Y$ の後に加える「ゲートノイズ」）、(c) 読み出し誤差の 3 種類です。 -->

We add three kinds of noise: (a) depolarizing error, (b) thermal relaxation error ((a) and (b) are composed into the "gate noise" added after $R_X$ and $R_Y$), and (c) readout error.

<!--
**ノイズが結果に与える影響の予想**：
- このモデルでは、ノイズは量子ビットごとに独立に加わり、量子ビット間の干渉（crosstalk）は入っていません。したがって、各量子ビットの振る舞いは C1 回路とまったく同じで、bias は統計的な揺らぎの範囲に収まると予想されます。
- 量子ビット間の相関も生じないので、同時分布は一様分布 $1/32$ から揺らぎの範囲でしかずれず、相関係数もほぼ 0 になるはずです。
-->

**Expected effects of the noise on the results**:
- In this model, noise is applied independently to each qubit, and interference between qubits (crosstalk) is not included. Therefore, each qubit behaves exactly as in the C1 circuit, and the bias is expected to stay within the range of statistical fluctuation.
- No correlation between qubits arises either, so the joint distribution should deviate from the uniform distribution $1/32$ only within the range of fluctuation, and the correlation coefficients should be nearly 0.

<!-- **このモデルでは再現できない実機の性質**：論文は、Odra 5 の実機で C2 回路が全体の平均では最も 50% に近い一方、**量子ビットごとのばらつきが最も大きい**と報告しています（Table III）。全量子ビットをまとめた "flatten" の 1 の割合は 49.72% でした。また、全体に 1 の割合が 50% より低くなる傾向を、非対称な読み出し誤差 $P(0|1) > P(1|0)$ で説明し、校正データから C1・C2 では 49.09% と見積もっています。このノートブックのモデルは全量子ビットで同じ・対称な誤差を使うので、こうした量子ビットごとの違いや偏りは現れません。 -->

**Properties of real hardware that this model cannot reproduce**: The paper reports that on the real Odra 5 hardware, the C2 circuit is closest to 50% on overall average but has **the largest variation from qubit to qubit** (Table III). The fraction of 1s for "flatten," which combines all qubits, was 49.72%. The paper also explains the overall tendency for the fraction of 1s to fall below 50% by asymmetric readout error $P(0|1) > P(1|0)$, and estimates 49.09% for C1 and C2 from the calibration data. Because the model in this notebook uses the same symmetric error on all qubits, such qubit-to-qubit differences and biases do not appear.

<!-- このノートブックでは、ノイズの種類を次の 3 つから選べます。 -->

In this notebook, the type of noise can be chosen from the following three options.

<!--
| `NOISE_KIND` | 加えるノイズ |
|---|---|
| `"gate"` | (a) + (b) のゲートノイズだけ |
| `"readout"` | (c) の読み出し誤差だけ |
| `"both"` | ゲートノイズと読み出し誤差の両方 |
-->

| `NOISE_KIND` | Noise added |
|---|---|
| `"gate"` | Only the gate noise (a) + (b) |
| `"readout"` | Only the readout error (c) |
| `"both"` | Both the gate noise and the readout error |

---

<!-- ### F. 使用する Qiskit の関数・クラス -->

### F. Qiskit functions and classes used

<!-- C1 のノートブックで使ったものに加えて、次のものを使います。 -->

In addition to those used in the C1 notebook, we use the following.

<!--
| 関数・クラス | パッケージ | 役割 |
|---|---|---|
| `QuantumCircuit(5, 5)` | `qiskit` | 量子ビット 5 個・古典ビット 5 個の回路を作る |
| `qc.h(q)`, `qc.rx(θ, q)`, `qc.ry(θ, q)` | `qiskit` | 量子ビット `q` にゲートを作用させる（`for` 文で 5 個に作用させる） |
| `qc.measure(range(5), range(5))` | `qiskit` | 量子ビット 0〜4 を測定し、古典ビット 0〜4 に記録する |
| `transpile(qc, basis_gates=...)` | `qiskit` | 回路を指定したゲートだけを使う回路に変換する |
| `qc_t.count_ops()` | `qiskit` | 回路に含まれるゲートの種類と個数を数える |
| `Operator(qc).equiv(...)` | `qiskit.quantum_info` | 2 つの回路が（大域位相を除いて）同じ演算か確かめる |
| `AerSimulator(noise_model=...)` | `qiskit_aer` | 量子回路のシミュレータ。ノイズモデルを渡せる |
| `AerSimulator(method="density_matrix")` | `qiskit_aer` | 密度行列でシミュレーションする（5.9 でノイズの影響を正確に計算するのに使う） |
| `qc.save_density_matrix()` | `qiskit_aer` | 測定直前の密度行列を結果に保存する（5.9） |
| `result.get_counts()` | `qiskit` | 5 ビットの結果ごとの回数の辞書（例 `{'00000': 6232, '00001': 6272, ...}`）を返す |
| `result.get_memory()` | `qiskit` | 各 shot の 5 ビットの文字列を測定順のリストで返す（生のデータ） |
| `NoiseModel`, `depolarizing_error`, `thermal_relaxation_error`, `ReadoutError` | `qiskit_aer.noise` | ノイズモデルを作る（C1 と同じ） |
| `noise_model.add_all_qubit_quantum_error(err, ["rx", "ry"])` | `qiskit_aer.noise` | 全量子ビットの指定したゲートの後に誤差を加える |
| `noise_model.add_all_qubit_readout_error(err)` | `qiskit_aer.noise` | 全量子ビットの測定に読み出し誤差を加える |
-->

| Function / class | Package | Role |
|---|---|---|
| `QuantumCircuit(5, 5)` | `qiskit` | Creates a circuit with 5 qubits and 5 classical bits |
| `qc.h(q)`, `qc.rx(θ, q)`, `qc.ry(θ, q)` | `qiskit` | Applies a gate to qubit `q` (applied to all 5 with a `for` loop) |
| `qc.measure(range(5), range(5))` | `qiskit` | Measures qubits 0–4 and records the results in classical bits 0–4 |
| `transpile(qc, basis_gates=...)` | `qiskit` | Converts a circuit into one that uses only the specified gates |
| `qc_t.count_ops()` | `qiskit` | Counts the types and numbers of gates in the circuit |
| `Operator(qc).equiv(...)` | `qiskit.quantum_info` | Checks whether two circuits implement the same operation (up to a global phase) |
| `AerSimulator(noise_model=...)` | `qiskit_aer` | Quantum circuit simulator. A noise model can be passed in |
| `AerSimulator(method="density_matrix")` | `qiskit_aer` | Simulates with density matrices (used in 5.9 to compute the effect of noise exactly) |
| `qc.save_density_matrix()` | `qiskit_aer` | Saves the density matrix just before measurement to the result (5.9) |
| `result.get_counts()` | `qiskit` | Returns a dictionary of counts for each 5-bit outcome (e.g., `{'00000': 6232, '00001': 6272, ...}`) |
| `result.get_memory()` | `qiskit` | Returns the 5-bit string of each shot as a list in measurement order (the raw data) |
| `NoiseModel`, `depolarizing_error`, `thermal_relaxation_error`, `ReadoutError` | `qiskit_aer.noise` | Build the noise model (same as C1) |
| `noise_model.add_all_qubit_quantum_error(err, ["rx", "ry"])` | `qiskit_aer.noise` | Adds an error after the specified gates on all qubits |
| `noise_model.add_all_qubit_readout_error(err)` | `qiskit_aer.noise` | Adds a readout error to the measurement of all qubits |

<!-- 数値計算には `numpy`、グラフには `matplotlib` を使います。 -->

`numpy` is used for numerical computation and `matplotlib` for plots.

---

<!-- ### G. このノートブックの使い方 -->

### G. How to use this notebook

<!-- 上のセルから順に実行してください。シミュレーションの条件は、次の変数を書き換えて選びます。 -->

Run the cells in order from the top. The simulation conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 選択肢・既定値 |
|---|---|---|
| `SAVE_RESULTS` | 4.3〜5.8 の raw data と結果を保存するか（1. のセル） | `True`（`results/c2/<日時>/` に保存）または `False`（保存しない。既定） |
| `SAVE_SWEEP` | 5.9 のノイズの強さを変える実験の raw data と結果を保存するか（1. のセル） | `True`（同じ日時のフォルダに保存）または `False`（保存しない。既定） |
| `GATE_IDEAL` | ノイズなしで使うゲート | `"H"`, `"Rx"`, `"Ry"`（既定 `"H"`） |
| `SHOTS_IDEAL` | ノイズなしの測定回数 | 既定 `200_000`（連結すると $10^6$ ビット） |
| `GATE_NOISY` | ノイズありで使うゲート | `"H"`, `"Rx"`, `"Ry"`（既定 `"H"`） |
| `NOISE_KIND` | 加えるノイズの種類 | `"gate"`, `"readout"`, `"both"`（既定 `"both"`） |
| `SHOTS_NOISY` | ノイズありの測定回数 | 既定 `200_000` |
| `N_TRIALS` | 繰り返しの回数 | 既定 `10` |
| `SEED` | シミュレータの乱数シード | `None`（毎回違う結果）または整数（結果を再現できる） |
-->

| Variable | Meaning | Options / default |
|---|---|---|
| `SAVE_RESULTS` | Whether to save the raw data and results of 4.3–5.8 (cell in 1.) | `True` (save in `results/c2/<date_time>/`) or `False` (do not save; default) |
| `SAVE_SWEEP` | Whether to save the raw data and results of the noise-strength sweep in 5.9 (cell in 1.) | `True` (save in the same date-and-time folder) or `False` (do not save; default) |
| `GATE_IDEAL` | Gate used in the noiseless case | `"H"`, `"Rx"`, `"Ry"` (default `"H"`) |
| `SHOTS_IDEAL` | Number of noiseless measurements | Default `200_000` ($10^6$ bits when concatenated) |
| `GATE_NOISY` | Gate used in the noisy case | `"H"`, `"Rx"`, `"Ry"` (default `"H"`) |
| `NOISE_KIND` | Type of noise added | `"gate"`, `"readout"`, `"both"` (default `"both"`) |
| `SHOTS_NOISY` | Number of noisy measurements | Default `200_000` |
| `N_TRIALS` | Number of repetitions | Default `10` |
| `SEED` | Random seed of the simulator | `None` (different results each time) or an integer (results are reproducible) |

<!-- 測定回数の既定値は、連結したビット列が論文と同じ $10^6$ ビットになるように $2 \times 10^5$ shots にしています。 -->

The default number of measurements is $2 \times 10^5$ shots so that the concatenated bit string has $10^6$ bits, the same as in the paper.

<!-- #### 保存されるファイル -->

#### Saved files

<!-- 1. のセルで `SAVE_RESULTS = True` または `SAVE_SWEEP = True` にすると、そのセルを実行した日時の名前のフォルダ `results/c2/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作ります。`SAVE_RESULTS = True` なら 4.3〜5.8 の raw data と結果を、`SAVE_SWEEP = True` なら 5.9 の raw data と結果を、その中に保存します。5.9 は保存量が多いので、別のスイッチにしています。どちらも `False`（既定）なら何も保存しません。1. のセルを実行し直すたびに新しいフォルダになるので、前の結果を上書きすることはありません。 -->

If you set `SAVE_RESULTS = True` or `SAVE_SWEEP = True` in the cell in 1., a folder named after the date and time that cell was run, `results/c2/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created. `SAVE_RESULTS = True` saves the raw data and results of 4.3–5.8 in it, and `SAVE_SWEEP = True` saves those of 5.9. Because 5.9 produces a lot of data, it has its own switch. If both are `False` (the default), nothing is saved. Each time you rerun the cell in 1., a new folder is used, so earlier results are never overwritten.

<!-- `RUN_DIR` の中は、実験（節）ごとのフォルダに分かれています。フォルダ名の先頭の数字は、ノートブックの節の番号です。 -->

Inside `RUN_DIR`, there is one folder per experiment (section). The number at the start of each folder name is the section number in this notebook.

<!--
| フォルダ | 節 | 中身 |
|---|---|---|
| `4.3_ideal_single/` | 4.3 | ノイズなしの 1 回の測定の raw data |
| `4.5_ideal_trials/` | 4.5 | ノイズなしの繰り返し測定の raw data（試行ごとに 1 ファイル）と、全試行の一覧 `trials.csv` |
| `4.6_ideal_gates/` | 4.6 | 3 つのゲートの raw data と、その一覧 `summary.csv` |
| `5.5_noisy_single/` | 5.5 | ノイズありの 1 回の測定の raw data |
| `5.7_noisy_trials/` | 5.7 | ノイズありの繰り返し測定の raw data（試行ごとに 1 ファイル）と `trials.csv` |
| `5.8_noisy_comparison/` | 5.8 | ゲート × ノイズの種類の 9 通りの raw data と `summary.csv` |
| `5.9_noise_sweep/` | 5.9 | ノイズの強さを変えた各条件・各試行の raw data（ノイズの種類ごとのフォルダ）と、`trials.csv`・`summary.csv`（`SAVE_SWEEP = True` のときだけ） |
-->

| Folder | Section | Contents |
|---|---|---|
| `4.3_ideal_single/` | 4.3 | Raw data of the single noiseless measurement |
| `4.5_ideal_trials/` | 4.5 | Raw data of the repeated noiseless measurements (one file per trial) and the list of all trials, `trials.csv` |
| `4.6_ideal_gates/` | 4.6 | Raw data for the three gates and their list, `summary.csv` |
| `5.5_noisy_single/` | 5.5 | Raw data of the single noisy measurement |
| `5.7_noisy_trials/` | 5.7 | Raw data of the repeated noisy measurements (one file per trial) and `trials.csv` |
| `5.8_noisy_comparison/` | 5.8 | Raw data for the 9 combinations of gate × noise type and `summary.csv` |
| `5.9_noise_sweep/` | 5.9 | Raw data for each condition and trial with varying noise strength (one folder per noise type), `trials.csv`, and `summary.csv` (only when `SAVE_SWEEP = True`) |

<!-- 既定の設定で両方を保存すると、次のようになります。 -->

With the default settings and both switches on, the saved files look like this.

<!--
```
results/c2/
└── 2026-11-10_153012/                 RUN_DIR（1. のセルを実行した日時）
    ├── 4.3_ideal_single/
    │   ├── c2_H_ideal.txt
    │   └── c2_H_ideal.txt.json
    ├── 4.5_ideal_trials/
    │   ├── c2_H_ideal_trial001.txt
    │   ├── c2_H_ideal_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 4.6_ideal_gates/
    │   ├── c2_H_ideal.txt
    │   ├── c2_H_ideal.txt.json
    │   ├── c2_Rx_ideal.txt ...
    │   └── summary.csv
    ├── 5.5_noisy_single/
    │   ├── c2_H_noise-both.txt
    │   └── c2_H_noise-both.txt.json
    ├── 5.7_noisy_trials/
    │   ├── c2_H_noise-both_trial001.txt
    │   ├── c2_H_noise-both_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 5.8_noisy_comparison/
    │   ├── c2_H_noise-gate.txt
    │   ├── c2_H_noise-gate.txt.json
    │   ├── c2_H_noise-readout.txt ...
    │   └── summary.csv
    └── 5.9_noise_sweep/
        ├── trials.csv
        ├── summary.csv
        ├── depolarizing/
        │   ├── c2_H_depolarizing-0_trial001.txt
        │   ├── c2_H_depolarizing-0_trial001.txt.json
        │   └── c2_H_depolarizing-0.01_trial001.txt ...
        └── ...
```
-->

```
results/c2/
└── 2026-11-10_153012/                 RUN_DIR (date and time the cell in 1. was run)
    ├── 4.3_ideal_single/
    │   ├── c2_H_ideal.txt
    │   └── c2_H_ideal.txt.json
    ├── 4.5_ideal_trials/
    │   ├── c2_H_ideal_trial001.txt
    │   ├── c2_H_ideal_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 4.6_ideal_gates/
    │   ├── c2_H_ideal.txt
    │   ├── c2_H_ideal.txt.json
    │   ├── c2_Rx_ideal.txt ...
    │   └── summary.csv
    ├── 5.5_noisy_single/
    │   ├── c2_H_noise-both.txt
    │   └── c2_H_noise-both.txt.json
    ├── 5.7_noisy_trials/
    │   ├── c2_H_noise-both_trial001.txt
    │   ├── c2_H_noise-both_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 5.8_noisy_comparison/
    │   ├── c2_H_noise-gate.txt
    │   ├── c2_H_noise-gate.txt.json
    │   ├── c2_H_noise-readout.txt ...
    │   └── summary.csv
    └── 5.9_noise_sweep/
        ├── trials.csv
        ├── summary.csv
        ├── depolarizing/
        │   ├── c2_H_depolarizing-0_trial001.txt
        │   ├── c2_H_depolarizing-0_trial001.txt.json
        │   └── c2_H_depolarizing-0.01_trial001.txt ...
        └── ...
```

<!--
raw data のファイル名は `<回路>_<ゲート>_<条件>[_trial<試行番号>].txt` です。

- 回路：`c2`
- ゲート：`H`, `Rx`, `Ry`
- 条件：`ideal`（ノイズなし）、`noise-gate`・`noise-readout`・`noise-both`（5.3 の `NOISE_KIND`）、5.9 では `<ノイズの種類>-<強さ>`（例：`readout-0.04`、`thermal-2500`）。5.9 のノイズの種類は `depolarizing`, `thermal`, `readout` です
- 試行番号：繰り返し測定（4.5・5.7・5.9）だけにつく、1 から数えた 3 桁の番号
-->

Raw-data files are named `<circuit>_<gate>_<condition>[_trial<trial number>].txt`.

- Circuit: `c2`
- Gate: `H`, `Rx`, `Ry`
- Condition: `ideal` (no noise); `noise-gate`, `noise-readout`, `noise-both` (`NOISE_KIND` in 5.3); in 5.9, `<noise type>-<strength>` (e.g., `readout-0.04`, `thermal-2500`). The noise types in 5.9 are `depolarizing`, `thermal`, `readout`
- Trial number: only for repeated measurements (4.5, 5.7, 5.9); a 3-digit number counted from 1

<!--
| ファイル | 中身 |
|---|---|
| `<名前>.txt` | `'0'` と `'1'` の文字を改行なしで並べたテキスト（長さ $5N$）。各 shot の 5 文字を Qiskit の順（量子ビット 4, 3, 2, 1, 0）のまま、測定順につなげたもの。SP800_90B・Toeplitz・Trevisan・SP800_22 のノートブックの `load_bits` で読める |
| `<名前>.txt.json` | 測定の条件と結果。キーは `circuit`, `gate`, `shots`, `seed`, `metadata`（条件・試行番号など）, `n_bits`, `bit_order`, `basis_gates`, `optimization_level`, `method`, `qiskit_version`, `qiskit_aer_version`, `noise_model`（`NoiseModel.to_dict` の内容。ノイズなしなら `null`）, `counts`, `evaluation`（`evaluate_c2` の戻り値） |
| `trials.csv`（4.5・5.7） | 1 試行 1 行。列は `file`（raw data のファイル名）, `gate`, `condition`, `trial`, `seed` と、`evaluate_c2` の戻り値のうち数値の項目 |
| `summary.csv`（4.6・5.8） | 1 ファイル 1 行。列は `file`, `gate`, `condition`, `seed` と、`evaluate_c2` の戻り値のうち数値の項目 |
| `5.9_noise_sweep/trials.csv` | 1 条件・1 試行 1 行。列は `component`, `strength`, `gate`, `trial`, `seed`, `shots`, `n_bits`, `P1`, `bias`, `shannon`, `min_entropy`, `joint_min_entropy_per_bit`, `joint_max_dev` |
| `5.9_noise_sweep/summary.csv` | 1 条件 1 行。列は `component`, `strength`, `gate`、密度行列から求めた `exact_P1`, `exact_bias`, `exact_min_entropy`, `exact_joint_min_entropy_per_bit`、`SWEEP_METRICS` の各指標の試行間の平均 `<指標>_mean` と標準偏差 `<指標>_std` |
-->

| File | Contents |
|---|---|
| `<name>.txt` | Text of the characters `'0'` and `'1'` with no line breaks (length $5N$): the 5 characters of each shot, in Qiskit order (qubits 4, 3, 2, 1, 0), joined in measurement order. Can be read with `load_bits` in the SP800_90B, Toeplitz, Trevisan, and SP800_22 notebooks |
| `<name>.txt.json` | Measurement conditions and results. Keys: `circuit`, `gate`, `shots`, `seed`, `metadata` (condition, trial number, and so on), `n_bits`, `bit_order`, `basis_gates`, `optimization_level`, `method`, `qiskit_version`, `qiskit_aer_version`, `noise_model` (the contents of `NoiseModel.to_dict`; `null` without noise), `counts`, `evaluation` (the return value of `evaluate_c2`) |
| `trials.csv` (4.5, 5.7) | One row per trial. Columns: `file` (the raw-data file name), `gate`, `condition`, `trial`, `seed`, and the numeric items of the return value of `evaluate_c2` |
| `summary.csv` (4.6, 5.8) | One row per file. Columns: `file`, `gate`, `condition`, `seed`, and the numeric items of the return value of `evaluate_c2` |
| `5.9_noise_sweep/trials.csv` | One row per condition and trial. Columns: `component`, `strength`, `gate`, `trial`, `seed`, `shots`, `n_bits`, `P1`, `bias`, `shannon`, `min_entropy`, `joint_min_entropy_per_bit`, `joint_max_dev` |
| `5.9_noise_sweep/summary.csv` | One row per condition. Columns: `component`, `strength`, `gate`; `exact_P1`, `exact_bias`, `exact_min_entropy`, `exact_joint_min_entropy_per_bit` computed from the density matrix; and the mean `<metric>_mean` and standard deviation `<metric>_std` over trials of each of the `SWEEP_METRICS` |

<!--
- ファイルは上書きしません。同じ `RUN_DIR` のまま同じセルをもう一度実行すると、測定を始める前にエラー（`FileExistsError`）になります。もう一度保存するときは、1. のセルを実行し直して新しいフォルダを作ってください。
- CSV は、その実験のすべての試行・条件が終わったあとに、raw data と同じフォルダに書きます。CSV がないフォルダは、途中で止まった実験の raw data です。
- `run_c2(..., raw_path=...)` や `run_trials(..., raw_dir=...)` に自分で保存先を指定した場合は、`SAVE_RESULTS` に関係なくその場所に保存します（4.5 を参照）。
- 保存量の目安：1 ファイル約 1 MB（$2 \times 10^5$ shots × 5 ビット）です。既定の設定では、4.3〜5.8 の合計が約 34 MB、5.9（`SAVE_SWEEP = True` のとき）が約 280 MB です。
-->

- Files are never overwritten. If you run the same cell again with the same `RUN_DIR`, an error (`FileExistsError`) is raised before any measurement. To save again, rerun the cell in 1. to create a new folder.
- The CSV files are written in the same folder as the raw data, after all trials and conditions of that experiment have finished. A folder without CSV files holds the raw data of an experiment that stopped partway.
- If you give your own destination to `run_c2(..., raw_path=...)` or `run_trials(..., raw_dir=...)`, the files are saved there regardless of `SAVE_RESULTS` (see 4.5).
- Approximate size: Each file is about 1 MB ($2 \times 10^5$ shots × 5 bits). With the default settings, 4.3–5.8 save about 34 MB in total, and 5.9 (when `SAVE_SWEEP = True`) saves about 280 MB.

<!-- ## 1. 準備 -->

## 1. Setup

<!-- 使うライブラリを読み込み、ネイティブゲート（説明パート D）を `BASIS_GATES` として定義します。 -->

We load the libraries used and define the native gates (Explanation Part D) as `BASIS_GATES`.

<!--
- `qiskit`：回路を作る・transpile する
- `qiskit_aer`：シミュレータとノイズモデル
- `numpy`：同時分布や相関係数の計算
- `IPython.display`：結果の表を Markdown で表示する
- `matplotlib`：4.4 と 5.9 のグラフを描く
-->

- `qiskit`: building and transpiling circuits
- `qiskit_aer`: simulator and noise models
- `numpy`: computing joint distributions and correlation coefficients
- `IPython.display`: displaying result tables as Markdown
- `matplotlib`: drawing the plots in 4.4 and 5.9

<!-- `ALL_BITSTRINGS` は 5 ビットの結果 32 通りを `'00000'` から `'11111'` の順に並べたリストです。 -->

`ALL_BITSTRINGS` is a list of the 32 possible 5-bit outcomes in order from `'00000'` to `'11111'`.

<!-- **結果の保存**：このセルの 2 つのスイッチで、raw data と結果を保存するかどうかを選びます。`SAVE_RESULTS = True` で 4.3〜5.8 を、`SAVE_SWEEP = True` で 5.9（ノイズの強さを段階的に変える実験。保存量が多い）を保存します。どちらかを `True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/c2/2026-11-10_153012/`）を作り、その中の実験ごとのフォルダに保存します。どちらも `False`（既定）なら何も保存しません。保存先と中身は「G. このノートブックの使い方」の「保存されるファイル」を参照してください。 -->

**Saving the results**: the two switches in this cell select whether the raw data and results are saved. `SAVE_RESULTS = True` saves 4.3–5.8, and `SAVE_SWEEP = True` saves 5.9 (the experiment that changes the noise strength step by step, which produces a lot of data). If either is `True`, a folder named after the date and time this cell was run (e.g., `results/c2/2026-11-10_153012/`) is created, and the files are saved in per-experiment folders inside it. If both are `False` (the default), nothing is saved. For the destinations and contents, see "Saved files" in "G. How to use this notebook".

<!-- セルは「回路の定数」「シミュレーションのシード」「保存のスイッチ」の 3 つのまとまりに分けています。保存に使う関数は、次の「保存に使う関数」のセルにまとめています。 -->

The cell is divided into three groups: circuit constants, the simulation seed, and the saving switches. The functions used for saving are collected in the following "Functions used for saving" cell.

In [ ]:
import csv
import json
import math
import statistics
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qiskit
import qiskit_aer
from IPython.display import Markdown, display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error, thermal_relaxation_error

print("qiskit     :", qiskit.__version__)
print("qiskit-aer :", qiskit_aer.__version__)

# ---- Circuit ----
BASIS_GATES = ["rx", "ry", "cz"]    # Native gates of IQM Spark (Odra 5) (arXiv:2512.09862, Table II)
GATES = ["H", "Rx", "Ry"]           # Gates that create the superposition
N_QUBITS = 5
ALL_BITSTRINGS = [format(k, f"0{N_QUBITS}b") for k in range(2 ** N_QUBITS)]   # the 32 outcomes (rightmost = qubit 0)

# ---- Simulation ----
SEED = None    # Random seed of the simulator (None: different results each time; integer: reproducible)

# ---- Saving (see "Saved files" in G) ----
SAVE_RESULTS = False    # Save the raw data and results of 4.3-5.8
SAVE_SWEEP = False      # Save the raw data and results of the noise-strength sweep in 5.9 (large)
RESULTS_ROOT = Path("results/c2")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if (SAVE_RESULTS or SAVE_SWEEP) else None

if RUN_DIR is None:
    print("save to    : nothing is saved")
else:
    print(f"save to    : {RUN_DIR} (4.3-5.8: {SAVE_RESULTS}, 5.9: {SAVE_SWEEP})")

<!-- **保存に使う関数** -->

**Functions used for saving**

<!-- raw data と CSV の保存は、次のセルの関数にまとめています。`run_c2`（4.5）や 4.3・5.5 のセルはこれらを呼び出すだけです。どのファイルも上書きしません。 -->

Saving the raw data and the CSV files is collected in the functions of the next cell. `run_c2` (4.5) and the cells in 4.3 and 5.5 only call them. No file is ever overwritten.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `output_dir` | 4.3〜5.8 の実験のフォルダ `RUN_DIR/<name>`（`SAVE_RESULTS = False` なら `None`） | `output_dir("4.5_ideal_trials")` |
| `raw_files` | 1 組の raw data の 2 つのファイルのパス（`.txt`, `.txt.json`） | `raw_files("results/c2/manual/c2_H_ideal.txt")` |
| `ensure_new` | どちらかのファイルがすでにあればエラー（測定の前に確かめるため） | `ensure_new("results/c2/manual/c2_H_ideal.txt")` |
| `save_raw` | 各 shot の測定結果（各 shot の 5 文字を連結）のビット列と条件の JSON を保存し、`.txt` のパスを返す | `save_raw(path, result.get_memory(), "H", 1000, 2026)` |
| `write_csv` | 1 レコード 1 行の CSV を書き、パスを返す（数値と文字列の項目だけを列にする） | `write_csv(path, records)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `output_dir` | Folder of an experiment in 4.3–5.8, `RUN_DIR/<name>` (`None` when `SAVE_RESULTS = False`) | `output_dir("4.5_ideal_trials")` |
| `raw_files` | Paths of the two files of one raw-data set (`.txt`, `.txt.json`) | `raw_files("results/c2/manual/c2_H_ideal.txt")` |
| `ensure_new` | Raises an error if either file already exists (checked before measuring) | `ensure_new("results/c2/manual/c2_H_ideal.txt")` |
| `save_raw` | Saves the bit string of the measurement results (the 5 characters of each shot joined) and the JSON of conditions, and returns the path of the `.txt` | `save_raw(path, result.get_memory(), "H", 1000, 2026)` |
| `write_csv` | Writes a CSV with one row per record and returns its path (only numeric and string items become columns) | `write_csv(path, records)` |

<!-- `save_raw` の引数 `gate`, `shots`, `seed`, `noise_model`, `method`, `optimization_level`, `metadata` は、JSON に記録する測定の条件です。 -->

The arguments `gate`, `shots`, `seed`, `noise_model`, `method`, `optimization_level`, and `metadata` of `save_raw` are the measurement conditions recorded in the JSON.

In [ ]:
def output_dir(name):
    # Folder of one experiment in 4.3-5.8 (None when SAVE_RESULTS = False)
    return RUN_DIR / name if SAVE_RESULTS else None


def raw_files(path):
    # The two files of one raw-data set: the bit string and the conditions
    path = Path(path)
    return path, path.with_name(path.name + ".json")


def ensure_new(path):
    # Files are never overwritten: stop if either file already exists
    for p in raw_files(path):
        if p.exists():
            raise FileExistsError(f"Destination already exists: {p}. Run the Setup cell (1.) again to save in a new folder.")


def save_raw(path, memory, gate, shots, seed, noise_model=None, *,
             method="automatic", optimization_level=None, metadata=None):
    # memory: the 5-bit string of each shot in measurement order (result.get_memory())
    ensure_new(path)
    bits_path, info_path = raw_files(path)
    counts = dict(Counter(memory))
    info = {
        "circuit": "C2", "gate": gate, "shots": shots, "seed": seed, "metadata": metadata,
        "n_bits": N_QUBITS * shots,
        "bit_order": "each shot: q4 q3 q2 q1 q0 (Qiskit order), shots concatenated in measurement order",
        "basis_gates": BASIS_GATES, "optimization_level": optimization_level, "method": method,
        "qiskit_version": qiskit.__version__, "qiskit_aer_version": qiskit_aer.__version__,
        "noise_model": None if noise_model is None else noise_model.to_dict(serializable=True),
        "counts": counts, "evaluation": evaluate_c2(counts),
    }
    bits_path.parent.mkdir(parents=True, exist_ok=True)
    bits_path.write_text("".join(memory), encoding="ascii")
    info_path.write_text(json.dumps(info, indent=2) + "\n", encoding="utf-8")
    return bits_path


def write_csv(path, records):
    # One row per record; only scalar values become columns. Existing files are never overwritten.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = [k for k, v in records[0].items() if isinstance(v, (str, int, float, bool, type(None)))]
    with path.open("x", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
    return path

<!-- ## 2. 評価指標を計算する関数 -->

## 2. Functions for computing the evaluation metrics

<!-- 説明パート C の 3 つの見方を関数にします。 -->

We turn the three viewpoints of Explanation Part C into functions.

<!--
- `shannon_entropy(probs)`：確率のリストから $H_{\mathrm{Shannon}} = -\sum p \log_2 p$ を計算する。$p = 0$ の項は 0 として扱います。C1 では引数が `(p0, p1)` の 2 つでしたが、ここでは 32 通りの分布にも使えるよう、確率のリストを受け取ります。
- `min_entropy(probs)`：$H_{\min} = -\log_2 \max p$
- `qubit_value(bitstring, q)`：Qiskit の文字列から量子ビット $q$ の値（`"0"` または `"1"`）を取り出す。右端が量子ビット 0 です（説明パート B）。
- `evaluate_c2(counts)`：5 ビットの測定回数の辞書から、(1) 量子ビットごとの $\hat{P}_q(1)$・bias・相関係数、(2) 同時分布の $\hat{P}(x)$・$H_{\mathrm{Shannon}}$・$H_{\min}$、(3) 連結したビット列の $\hat{P}(1)$・bias・$H_{\mathrm{Shannon}}$・$H_{\min}$ をまとめて計算する
- `expected_bias(n_bits)`：公平なビットを `n_bits` 個測ったときの bias の目安 $\sqrt{2/\pi}\,/\,(2\sqrt{n})$
- `expected_joint_min_entropy(shots)`：理想的な一様分布から `shots` 回抽出したときの、同時分布の $H_{\min}$ の目安（多項分布から何度も抽出して平均する）
-->

- `shannon_entropy(probs)`: computes $H_{\mathrm{Shannon}} = -\sum p \log_2 p$ from a list of probabilities. Terms with $p = 0$ are treated as 0. In C1 there were two arguments `(p0, p1)`, but here it takes a list of probabilities so that it can also be used for the 32-outcome distribution.
- `min_entropy(probs)`: $H_{\min} = -\log_2 \max p$
- `qubit_value(bitstring, q)`: extracts the value of qubit $q$ (`"0"` or `"1"`) from a Qiskit string. The rightmost character is qubit 0 (Explanation Part B).
- `evaluate_c2(counts)`: computes together, from the 5-bit measurement-count dictionary, (1) the per-qubit $\hat{P}_q(1)$, bias, and correlation coefficients, (2) the joint distribution's $\hat{P}(x)$, $H_{\mathrm{Shannon}}$, and $H_{\min}$, and (3) the concatenated bit string's $\hat{P}(1)$, bias, $H_{\mathrm{Shannon}}$, and $H_{\min}$
- `expected_bias(n_bits)`: the reference bias $\sqrt{2/\pi}\,/\,(2\sqrt{n})$ when `n_bits` fair bits are measured
- `expected_joint_min_entropy(shots)`: the reference $H_{\min}$ of the joint distribution when sampling `shots` times from the ideal uniform distribution (obtained by sampling repeatedly from a multinomial distribution and averaging)

<!-- 連結したビット列の $\hat{P}(1)$ は、5 個の量子ビットの $\hat{P}_q(1)$ の平均と同じです。また、量子ビットの値の組は `counts` から数えられるので、相関係数を求めるのに `get_memory()` の生データは必要ありません。 -->

$\hat{P}(1)$ of the concatenated bit string equals the average of the 5 qubits' $\hat{P}_q(1)$. Also, the combinations of qubit values can be counted from `counts`, so the raw data from `get_memory()` is not needed to compute the correlation coefficients.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `shannon_entropy` | Shannon エントロピー（float、単位はビット） | `shannon_entropy([0.5, 0.5])` |
| `min_entropy` | min-entropy（float、単位はビット） | `min_entropy([1/32] * 32)` |
| `qubit_value` | 量子ビット `q` の値（文字列 `"0"` または `"1"`） | `qubit_value("00010", 1)` |
| `evaluate_c2` | 3 つの見方の評価結果をまとめた辞書 | `evaluate_c2({"00000": 3, "11111": 1})` |
| `expected_bias` | 公平なビットの bias の期待値の近似（float） | `expected_bias(1_000_000)` |
| `expected_joint_min_entropy` | 一様分布から抽出したときの同時分布の min-entropy の平均（float、5 ビットあたり） | `expected_joint_min_entropy(200_000)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `shannon_entropy` | Shannon entropy (float, in bits) | `shannon_entropy([0.5, 0.5])` |
| `min_entropy` | Min-entropy (float, in bits) | `min_entropy([1/32] * 32)` |
| `qubit_value` | Value of qubit `q` (the string `"0"` or `"1"`) | `qubit_value("00010", 1)` |
| `evaluate_c2` | Dictionary collecting the evaluation results from the three viewpoints | `evaluate_c2({"00000": 3, "11111": 1})` |
| `expected_bias` | Approximate expected bias of fair bits (float) | `expected_bias(1_000_000)` |
| `expected_joint_min_entropy` | Average min-entropy of the joint distribution when sampling from the uniform distribution (float, per 5 bits) | `expected_joint_min_entropy(200_000)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>shannon_entropy</code></td><td><code>probs</code></td><td>確率のリスト。各値は 0〜1 で、合計が 1（例：<code>[0.5, 0.5]</code>、32 通りなら 32 個）</td><td>必須</td></tr>
<tr><td><code>min_entropy</code></td><td><code>probs</code></td><td>確率のリスト。合計が 1</td><td>必須</td></tr>
<tr><td rowspan="2"><code>qubit_value</code></td><td><code>bitstring</code></td><td>Qiskit の測定結果の文字列（例：<code>"00010"</code>）</td><td>必須</td></tr>
<tr><td><code>q</code></td><td>量子ビットの番号。0〜4 の整数</td><td>必須</td></tr>
<tr><td><code>evaluate_c2</code></td><td><code>counts</code></td><td>5 ビットの測定集計の辞書。キーは <code>"00000"</code>〜<code>"11111"</code>、値は非負の整数。合計は 1 以上。出なかったキーは省略可</td><td>必須</td></tr>
<tr><td><code>expected_bias</code></td><td><code>n_bits</code></td><td>ビット数。1 以上の整数（連結したビット列なら <code>5 * shots</code>）</td><td>必須</td></tr>
<tr><td rowspan="3"><code>expected_joint_min_entropy</code></td><td><code>shots</code></td><td>shot 数。1 以上の整数</td><td>必須</td></tr>
<tr><td><code>n_samples</code></td><td>平均をとる抽出の回数。大きいほど正確だが時間がかかる</td><td><code>500</code></td></tr>
<tr><td><code>seed</code></td><td>抽出に使う乱数のシード（整数）</td><td><code>0</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>shannon_entropy</code></td><td><code>probs</code></td><td>List of probabilities. Each value is from 0 to 1 and the total is 1 (e.g., <code>[0.5, 0.5]</code>; 32 values for the 32 outcomes)</td><td>Required</td></tr>
<tr><td><code>min_entropy</code></td><td><code>probs</code></td><td>List of probabilities. The total is 1</td><td>Required</td></tr>
<tr><td rowspan="2"><code>qubit_value</code></td><td><code>bitstring</code></td><td>Qiskit measurement-result string (e.g., <code>"00010"</code>)</td><td>Required</td></tr>
<tr><td><code>q</code></td><td>Qubit index. An integer from 0 to 4</td><td>Required</td></tr>
<tr><td><code>evaluate_c2</code></td><td><code>counts</code></td><td>Dictionary of 5-bit measurement tallies. Keys are <code>"00000"</code> to <code>"11111"</code>; values are non-negative integers with a total of at least 1. Keys that did not occur may be omitted</td><td>Required</td></tr>
<tr><td><code>expected_bias</code></td><td><code>n_bits</code></td><td>Number of bits. An integer of at least 1 (<code>5 * shots</code> for the concatenated bit string)</td><td>Required</td></tr>
<tr><td rowspan="3"><code>expected_joint_min_entropy</code></td><td><code>shots</code></td><td>Number of shots. An integer of at least 1</td><td>Required</td></tr>
<tr><td><code>n_samples</code></td><td>Number of samplings to average over. Larger is more accurate but takes longer</td><td><code>500</code></td></tr>
<tr><td><code>seed</code></td><td>Random seed used for sampling (integer)</td><td><code>0</code></td></tr>
</tbody>
</table>

<br>

<!-- `evaluate_c2` の戻り値の主なキーは次のとおりです。 -->

The main keys of the return value of `evaluate_c2` are as follows.

<!--
| キー | 意味 |
|---|---|
| `shots`, `n_bits` | shot 数 $N$ と、連結したビット列の長さ $5N$ |
| `n0`, `n1`, `P0`, `P1`, `bias`, `shannon`, `min_entropy` | (3) 連結したビット列の 0/1 の個数・確率・bias・エントロピー（1 ビットあたり） |
| `joint_shannon`, `joint_min_entropy` | (2) 同時分布のエントロピー（5 ビットあたり） |
| `joint_min_entropy_per_bit` | (2) の `joint_min_entropy` を 5 で割った値 |
| `joint_max_dev` | (2) $\max_x \lvert\hat{P}(x) - 1/32\rvert$ |
| `joint_counts` | (2) 32 通りの結果ごとの回数の辞書 |
| `qubit_P1`, `qubit_bias` | (1) 量子ビット 0〜4 の $\hat{P}_q(1)$ と bias のリスト |
| `corr` | (1) 相関係数 $r_{ij}$ の 5×5 のリスト |
-->

| Key | Meaning |
|---|---|
| `shots`, `n_bits` | Number of shots $N$ and length $5N$ of the concatenated bit string |
| `n0`, `n1`, `P0`, `P1`, `bias`, `shannon`, `min_entropy` | (3) Numbers of 0s and 1s, probabilities, bias, and entropies (per bit) of the concatenated bit string |
| `joint_shannon`, `joint_min_entropy` | (2) Entropies of the joint distribution (per 5 bits) |
| `joint_min_entropy_per_bit` | `joint_min_entropy` of (2) divided by 5 |
| `joint_max_dev` | (2) $\max_x \lvert\hat{P}(x) - 1/32\rvert$ |
| `joint_counts` | (2) Dictionary of counts for each of the 32 outcomes |
| `qubit_P1`, `qubit_bias` | (1) Lists of $\hat{P}_q(1)$ and bias for qubits 0–4 |
| `corr` | (1) 5×5 list of correlation coefficients $r_{ij}$ |


In [ ]:
def shannon_entropy(probs):
    # Treat 0 log 0 as 0
    return -sum(p * math.log2(p) for p in probs if p > 0)


def min_entropy(probs):
    return -math.log2(max(probs))


def qubit_value(bitstring, q):
    # In Qiskit strings, the rightmost character is qubit 0
    return bitstring[len(bitstring) - 1 - q]


def evaluate_c2(counts):
    shots = sum(counts.values())
    joint_counts = {x: counts.get(x, 0) for x in ALL_BITSTRINGS}
    joint_probs = [n / shots for n in joint_counts.values()]

    # Table of 0/1 for 32 outcomes × 5 qubits, and the count of each outcome
    bits = np.array([[int(qubit_value(x, q)) for q in range(N_QUBITS)] for x in ALL_BITSTRINGS])
    weights = np.array(list(joint_counts.values()), dtype=float)

    # (1) Per qubit
    p1 = bits.T @ weights / shots                    # Probability of 1 on each qubit
    p11 = (bits.T * weights) @ bits / shots          # Probability that both of two qubits are 1
    var = p1 * (1 - p1)
    with np.errstate(divide="ignore", invalid="ignore"):
        corr = (p11 - np.outer(p1, p1)) / np.sqrt(np.outer(var, var))

    # (3) Concatenated bit string (5.9 passes probabilities as counts, so compute P(1) from n1 before rounding)
    n_bits = N_QUBITS * shots
    n1_value = float(p1.sum()) * shots
    n1 = int(round(n1_value))
    P1 = n1_value / n_bits
    P0 = 1 - P1

    return {
        "shots": shots,
        "n_bits": n_bits,
        "n0": n_bits - n1,
        "n1": n1,
        "P0": P0,
        "P1": P1,
        "bias": abs(P1 - 1 / 2),
        "shannon": shannon_entropy([P0, P1]),
        "min_entropy": min_entropy([P0, P1]),
        "joint_shannon": shannon_entropy(joint_probs),
        "joint_min_entropy": min_entropy(joint_probs),
        "joint_min_entropy_per_bit": min_entropy(joint_probs) / N_QUBITS,
        "joint_max_dev": max(abs(p - 1 / 2 ** N_QUBITS) for p in joint_probs),
        "joint_counts": joint_counts,
        "qubit_P1": [float(v) for v in p1],
        "qubit_bias": [float(abs(v - 1 / 2)) for v in p1],
        "corr": [[float(v) for v in row] for row in corr],
    }


def expected_bias(n_bits):
    # Expected bias when n_bits fair bits are measured (normal approximation)
    return math.sqrt(2 / math.pi) / (2 * math.sqrt(n_bits))


def expected_joint_min_entropy(shots, n_samples=500, seed=0):
    # Compute the min-entropy of sampling shots times from the uniform distribution (1/32 each) n_samples times and average
    rng = np.random.default_rng(seed)
    k = 2 ** N_QUBITS
    max_counts = rng.multinomial(shots, [1 / k] * k, size=n_samples).max(axis=1)
    return float(np.mean(-np.log2(max_counts / shots)))

<!-- 次に、評価結果を表やグラフで表示する関数を定義します。 -->

Next, we define functions that display the evaluation results as tables and plots.

<!--
- `show_qubits(r)`：(1) 量子ビットごとの $\hat{P}_q(1)$ と bias の表
- `show_correlation(r)`：(1) 量子ビット間の相関係数の表
- `plot_joint(r)`：(2) 32 通りの結果の $\hat{P}(x)$ の棒グラフ。理想値 $1/32$ と、揺らぎの目安（$\pm 2\sigma$）の線を重ねます
- `show_per_qubit(r)`・`show_joint(r, title)`・`show_metrics(r)`：(1) 量子ビットごと・(2) 5 ビットの同時分布・(3) 連結したビット列の評価を、目安の値とまとめて表示する（4.4・5.6 で使う）
- `show_table(rows, first_col)`：複数の評価結果を 1 つの表に並べる（4.5 以降で使う）
-->

- `show_qubits(r)`: table of (1) the per-qubit $\hat{P}_q(1)$ and bias
- `show_correlation(r)`: table of (1) the correlation coefficients between qubits
- `plot_joint(r)`: bar chart of (2) $\hat{P}(x)$ for the 32 outcomes, overlaid with lines for the ideal value $1/32$ and the reference fluctuation ($\pm 2\sigma$)
- `show_per_qubit(r)`, `show_joint(r, title)`, `show_metrics(r)`: display the evaluation of (1) each qubit, (2) the 5-bit joint distribution, and (3) the concatenated bit string, together with reference values (used in 4.4 and 5.6)
- `show_table(rows, first_col)`: lists multiple evaluation results in one table (used from 4.5 onward)

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `show_qubits` | 量子ビットごとの表を表示。戻り値は `None` | `show_qubits(evaluate_c2({"00000": 3, "11111": 1}))` |
| `show_correlation` | 相関係数の 5×5 の表を表示。戻り値は `None` | `show_correlation(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}))` |
| `plot_joint` | 32 通りの結果の棒グラフを表示。戻り値は `None` | `plot_joint(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}))` |
| `show_per_qubit` | 量子ビットごとの表と相関係数の表を、目安の値とともに表示。戻り値は `None` | `show_per_qubit(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}))` |
| `show_joint` | 32 通りの棒グラフと同時分布のエントロピーを、目安の値とともに表示。戻り値は `None` | `show_joint(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}), "test")` |
| `show_metrics` | 連結したビット列の指標と公平なビットの目安を表示。戻り値は `None` | `show_metrics(evaluate_c2({"00000": 3, "11111": 1}))` |
| `show_table` | Markdown の表を表示。戻り値は `None` | `show_table([("H", evaluate_c2({"00000": 3, "11111": 1}))], "gate")` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `show_qubits` | Displays the per-qubit table. Returns `None` | `show_qubits(evaluate_c2({"00000": 3, "11111": 1}))` |
| `show_correlation` | Displays the 5×5 table of correlation coefficients. Returns `None` | `show_correlation(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}))` |
| `plot_joint` | Displays a bar chart of the 32 outcomes. Returns `None` | `plot_joint(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}))` |
| `show_per_qubit` | Displays the per-qubit table and the correlation table, with reference values. Returns `None` | `show_per_qubit(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}))` |
| `show_joint` | Displays the bar chart of the 32 outcomes and the entropies of the joint distribution, with reference values. Returns `None` | `show_joint(evaluate_c2({x: 100 for x in ALL_BITSTRINGS}), "test")` |
| `show_metrics` | Displays the metrics of the concatenated bit string and reference values for fair bits. Returns `None` | `show_metrics(evaluate_c2({"00000": 3, "11111": 1}))` |
| `show_table` | Displays a Markdown table. Returns `None` | `show_table([("H", evaluate_c2({"00000": 3, "11111": 1}))], "gate")` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>show_qubits</code></td><td><code>r</code></td><td><code>evaluate_c2</code> の戻り値</td><td>必須</td></tr>
<tr><td><code>show_correlation</code></td><td><code>r</code></td><td><code>evaluate_c2</code> の戻り値</td><td>必須</td></tr>
<tr><td rowspan="2"><code>plot_joint</code></td><td><code>r</code></td><td><code>evaluate_c2</code> の戻り値</td><td>必須</td></tr>
<tr><td><code>title</code></td><td>グラフのタイトル（英語推奨）</td><td><code>"Joint distribution"</code></td></tr>
<tr><td><code>show_per_qubit</code></td><td><code>r</code></td><td><code>evaluate_c2</code> の戻り値</td><td>必須</td></tr>
<tr><td rowspan="2"><code>show_joint</code></td><td><code>r</code></td><td><code>evaluate_c2</code> の戻り値</td><td>必須</td></tr>
<tr><td><code>title</code></td><td>グラフの題（英語推奨）</td><td>必須</td></tr>
<tr><td><code>show_metrics</code></td><td><code>r</code></td><td><code>evaluate_c2</code> の戻り値</td><td>必須</td></tr>
<tr><td rowspan="2"><code>show_table</code></td><td><code>rows</code></td><td><code>(ラベル, evaluate_c2 の戻り値)</code> のリスト。例：<code>[("H", r)]</code></td><td>必須</td></tr>
<tr><td><code>first_col</code></td><td>表の先頭列の見出し文字列。例：<code>"gate"</code>、<code>"trial"</code></td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>show_qubits</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c2</code></td><td>Required</td></tr>
<tr><td><code>show_correlation</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c2</code></td><td>Required</td></tr>
<tr><td rowspan="2"><code>plot_joint</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c2</code></td><td>Required</td></tr>
<tr><td><code>title</code></td><td>Plot title (English recommended)</td><td><code>"Joint distribution"</code></td></tr>
<tr><td><code>show_per_qubit</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c2</code></td><td>Required</td></tr>
<tr><td rowspan="2"><code>show_joint</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c2</code></td><td>Required</td></tr>
<tr><td><code>title</code></td><td>Plot title (English recommended)</td><td>Required</td></tr>
<tr><td><code>show_metrics</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c2</code></td><td>Required</td></tr>
<tr><td rowspan="2"><code>show_table</code></td><td><code>rows</code></td><td>List of <code>(label, return value of evaluate_c2)</code>. E.g., <code>[("H", r)]</code></td><td>Required</td></tr>
<tr><td><code>first_col</code></td><td>Heading string for the first column of the table. E.g., <code>"gate"</code>, <code>"trial"</code></td><td>Required</td></tr>
</tbody>
</table>


In [ ]:
def show_qubits(r):
    lines = ["| qubit | counts (1) | P(1) | bias |", "|---|---|---|---|"]
    for q in range(N_QUBITS):
        n1 = round(r["qubit_P1"][q] * r["shots"])
        lines.append(f"| {q} | {n1} | {r['qubit_P1'][q]:.5f} | {r['qubit_bias'][q]:.5f} |")
    display(Markdown("\n".join(lines)))


def show_correlation(r):
    header = "| qubit | " + " | ".join(str(j) for j in range(N_QUBITS)) + " |"
    lines = [header, "|---" * (N_QUBITS + 1) + "|"]
    for i in range(N_QUBITS):
        cells = " | ".join("―" if i == j else f"{r['corr'][i][j]:+.5f}" for j in range(N_QUBITS))
        lines.append(f"| {i} | {cells} |")
    display(Markdown("\n".join(lines)))


def plot_joint(r, title="Joint distribution"):
    k = 2 ** N_QUBITS
    probs = [r["joint_counts"][x] / r["shots"] for x in ALL_BITSTRINGS]
    sigma = math.sqrt((1 / k) * (1 - 1 / k) / r["shots"])
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.bar(range(k), probs, color="C0")
    ax.axhline(1 / k, color="black", label="ideal 1/32")
    for s in (+2, -2):
        ax.axhline(1 / k + s * sigma, color="gray", linestyle=":",
                   label="1/32 ± 2σ" if s > 0 else None)
    ax.set_xticks(range(k))
    ax.set_xticklabels(ALL_BITSTRINGS, rotation=90, fontsize=8)
    ax.set_ylim(1 / k - 6 * sigma, 1 / k + 6 * sigma)
    ax.set(xlabel="measured bitstring (q4 ... q0)", ylabel="P(x)", title=title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def show_table(rows, first_col):
    # rows: list of (label, result of evaluate_c2)
    lines = [
        f"| {first_col} | P(1) | bias | Shannon entropy | min-entropy | joint min-entropy / 5 | max abs(P(x) - 1/32) |",
        "|---|---|---|---|---|---|---|",
    ]
    for label, r in rows:
        lines.append(
            f"| {label} | {r['P1']:.5f} | {r['bias']:.5f} | {r['shannon']:.5f} | {r['min_entropy']:.5f} "
            f"| {r['joint_min_entropy_per_bit']:.5f} | {r['joint_max_dev']:.6f} |"
        )
    display(Markdown("\n".join(lines)))
    print("P(1), bias, Shannon entropy, and min-entropy are values for the concatenated bit string (per bit).")


def show_per_qubit(r):
    # (1) P(1) and bias of each qubit, and the correlation coefficients between qubits
    show_qubits(r)
    print(f"[Reference] expected bias for fair bits = {expected_bias(r['shots']):.5f}")
    show_correlation(r)
    print(f"[Reference] fluctuation of the correlation coefficient if independent ≈ ±{1 / math.sqrt(r['shots']):.5f}")


def show_joint(r, title):
    # (2) Bar chart and entropies of the 5-bit joint distribution
    plot_joint(r, title=title)
    print(f"Shannon entropy = {r['joint_shannon']:.5f} (max 5)")
    print(f"min-entropy     = {r['joint_min_entropy']:.5f} (max 5, per bit {r['joint_min_entropy_per_bit']:.5f})")
    print(f"max |P(x) - 1/32| = {r['joint_max_dev']:.6f}")
    h_ref = expected_joint_min_entropy(r["shots"])
    print(f"[Reference] min-entropy when sampling from the uniform distribution {r['shots']:,} times"
          f" ≈ {h_ref:.5f} (per bit {h_ref / N_QUBITS:.5f})")


def show_metrics(r):
    # Metrics of the concatenated bit string, with reference values for fair bits
    for label, k in [("P(0)", "P0"), ("P(1)", "P1"), ("bias", "bias"),
                     ("Shannon entropy", "shannon"), ("min-entropy", "min_entropy")]:
        print(f"{label:15} = {r[k]:.6f}")
    b_ref = expected_bias(r["n_bits"])
    print(f"[Reference] expected bias for fair bits = {b_ref:.6f}")
    print(f"[Reference] min-entropy at that bias    = {-math.log2(1 / 2 + b_ref):.6f}")

<!-- ## 3. C2 回路を作る -->

## 3. Building the C2 circuit

<!-- 説明パート A・B の C2 回路を作る関数 `build_c2(gate)` を定義します。 -->

We define the function `build_c2(gate)`, which builds the C2 circuit of Explanation Parts A and B.

<!--
1. `QuantumCircuit(5, 5)` で量子ビット 5 個・古典ビット 5 個の回路を作る（量子ビットはすべて $|0\rangle$ から始まる）
2. 選んだゲート（$H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ のいずれか）を、`for` 文で量子ビット 0〜4 のそれぞれに作用させる
3. 量子ビット 0〜4 を測定し、結果を古典ビット 0〜4 に記録する
-->

1. Create a circuit with 5 qubits and 5 classical bits using `QuantumCircuit(5, 5)` (all qubits start in $|0\rangle$)
2. Apply the chosen gate (one of $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$) to each of qubits 0–4 with a `for` loop
3. Measure qubits 0–4 and record the results in classical bits 0–4

<!-- 3 つのゲートすべてについて、元の回路と transpile 後の回路、ゲートの個数を表示します。$H_{\mathrm{Gate}}$ だけが各量子ビットで $R_Y(\pi/2)$ と $R_X(\pi)$ の 2 個に分解されることを確認してください（説明パート D）。最後に `Operator(...).equiv(...)` で、変換の前後で回路が同じ演算になっていることを確かめます。 -->

For all three gates, we display the original circuit, the transpiled circuit, and the gate counts. Confirm that only $H_{\mathrm{Gate}}$ is decomposed on each qubit into the two gates $R_Y(\pi/2)$ and $R_X(\pi)$ (Explanation Part D). Finally, `Operator(...).equiv(...)` confirms that the circuit implements the same operation before and after the conversion.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_c2` | 測定を含む 5 量子ビット回路（QuantumCircuit） | `build_c2("H")` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_c2` | 5-qubit circuit including measurement (QuantumCircuit) | `build_c2("H")` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>build_c2</code></td><td><code>gate</code></td><td>大文字小文字を区別して <code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>build_c2</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code> (case-sensitive)</td><td>Required</td></tr>
</tbody>
</table>


In [ ]:
def build_c2(gate):
    if gate not in GATES:
        raise ValueError(f"gate must be one of {GATES} (got: {gate!r})")

    qc = QuantumCircuit(N_QUBITS, N_QUBITS)
    for q in range(N_QUBITS):
        if gate == "H":
            qc.h(q)
        elif gate == "Rx":
            qc.rx(math.pi / 2, q)
        else:  # "Ry"
            qc.ry(math.pi / 2, q)
    qc.measure(range(N_QUBITS), range(N_QUBITS))
    return qc


for gate in GATES:
    qc = build_c2(gate)
    qc_t = transpile(qc, basis_gates=BASIS_GATES)

    print(f"===== {gate} =====")
    print("[Original circuit]")
    print(qc.draw())
    print("[Transpiled circuit]")
    print(qc_t.draw())
    print("Gate counts:", dict(qc_t.count_ops()))

    # Check, excluding measurements, that the operation is the same before and after conversion
    same = Operator(qc.remove_final_measurements(inplace=False)).equiv(
        Operator(qc_t.remove_final_measurements(inplace=False))
    )
    print("Same operation before and after conversion:", same)
    print()

<!-- ## 4. ノイズなしのシミュレーション -->

## 4. Noiseless simulation

<!-- まずノイズのない理想的なシミュレータで C2 回路を測定します。理想的には 32 通りの結果がどれも $1/32$ の確率で現れますが、有限回の測定では統計的に揺らぎます（説明パート C）。 -->

First, we measure the C2 circuit on an ideal simulator with no noise. Ideally all 32 outcomes occur with probability $1/32$, but a finite number of measurements fluctuates statistically (Explanation Part C).

<!-- ### 4.1 ゲートと測定回数を選ぶ -->

### 4.1 Choosing the gate and the number of measurements

<!-- `GATE_IDEAL` に `"H"`, `"Rx"`, `"Ry"` のいずれかを、`SHOTS_IDEAL` に測定回数（shot 数）を指定します。連結したビット列の長さは `5 * SHOTS_IDEAL` になります。 -->

Set `GATE_IDEAL` to one of `"H"`, `"Rx"`, `"Ry"`, and set `SHOTS_IDEAL` to the number of measurements (shots). The length of the concatenated bit string is `5 * SHOTS_IDEAL`.

In [ ]:
GATE_IDEAL = "H"         # Choose from "H", "Rx", "Ry"
SHOTS_IDEAL = 200_000    # Number of measurements (5 times as many bits when concatenated)

<!-- ### 4.2 回路を作って transpile する -->

### 4.2 Building and transpiling the circuit

<!-- `build_c2` で回路を作り、`transpile` でネイティブゲート `BASIS_GATES` だけを使う回路に変換します。 -->

We build the circuit with `build_c2` and use `transpile` to convert it into a circuit that uses only the native gates `BASIS_GATES`.

In [ ]:
qc_ideal = build_c2(GATE_IDEAL)
qc_ideal_t = transpile(qc_ideal, basis_gates=BASIS_GATES)

print(qc_ideal_t.draw())

<!-- ### 4.3 測定する -->

### 4.3 Measuring

<!-- `AerSimulator()` をノイズなしで作り、`backend.run(..., shots=SHOTS_IDEAL, memory=True)` で測定します。 -->

We create a noiseless `AerSimulator()` and measure with `backend.run(..., shots=SHOTS_IDEAL, memory=True)`.

<!--
- `result.get_counts()`：32 通りの 5 ビットの結果ごとの回数
- `result.get_memory()`：各 shot の 5 ビットの文字列を測定順に並べたリスト。実機なら乱数の元になる**生のデータ（raw data）**に相当する模擬データです。シミュレータの結果は古典コンピュータの擬似乱数から作られ、シードで再現できるので、真の量子乱数ではなく、暗号には使えません。
-->

- `result.get_counts()`: the count for each of the 32 possible 5-bit outcomes
- `result.get_memory()`: a list of the 5-bit string of each shot in measurement order. This is simulated data corresponding to the **raw data** that would be the source of random numbers on real hardware. Simulator results are generated from a classical computer's pseudorandom numbers and are reproducible with a seed, so they are not true quantum random numbers and must not be used for cryptography.

<!-- 最初の 5 shot について、文字列と量子ビットごとの値を並べて、ビットの並び順（右端が量子ビット 0）を確認します。最後に、5 文字ずつ shot 順につなげた**連結したビット列**（説明パート C の (3)）を作り、先頭 50 ビットを表示します。 -->

For the first 5 shots, we list the string and the value of each qubit to confirm the bit ordering (the rightmost character is qubit 0). Finally, we create the **concatenated bit string** (item (3) of Explanation Part C), joining the 5 characters of each shot in shot order, and display its first 50 bits.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、このセルの raw dataを `4.3_ideal_single/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data of this cell is saved in `4.3_ideal_single/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
backend_ideal = AerSimulator(seed_simulator=SEED)

job = backend_ideal.run(qc_ideal_t, shots=SHOTS_IDEAL, memory=True)
result = job.result()

counts_ideal = result.get_counts()
raw_shots_ideal = result.get_memory()

print("Number of distinct outcomes =", len(counts_ideal), "(ideally all 32 outcomes appear)")
print()
print("shot | string | q0 q1 q2 q3 q4")
for i, s in enumerate(raw_shots_ideal[:5]):
    values = "  ".join(qubit_value(s, q) for q in range(N_QUBITS))
    print(f"{i:4d} | {s}  | {values}")

flat_bits_ideal = "".join(raw_shots_ideal)
print()
print("Concatenated bit string (first 50 bits) =", flat_bits_ideal[:50])
print("Length of the concatenated bit string =", len(flat_bits_ideal))

# Save the raw data (only when SAVE_RESULTS = True)
if SAVE_RESULTS:
    path = save_raw(output_dir("4.3_ideal_single") / f"c2_{GATE_IDEAL}_ideal.txt", raw_shots_ideal,
                    GATE_IDEAL, SHOTS_IDEAL, SEED, metadata={"condition": "ideal"})
    print("Saved:", path)

<!-- ### 4.4 評価指標を計算する -->

### 4.4 Computing the evaluation metrics

<!-- `evaluate_c2` で 3 つの見方の指標を計算し、順に表示します。 -->

`evaluate_c2` computes the metrics for the three viewpoints, and we display them in order.

<!--
1. **量子ビットごと**：各量子ビットの $\hat{P}_q(1)$ と bias の表と、量子ビット間の相関係数の表です。bias は揺らぎの目安 $\sqrt{2/\pi}\,/\,(2\sqrt{N})$ と、相関係数は $\pm 1/\sqrt{N}$ と比べます。
2. **同時分布**：32 通りの $\hat{P}(x)$ の棒グラフです。灰色の点線は $1/32 \pm 2\sigma$ で、ほとんどの棒がこの範囲に入れば揺らぎの範囲と言えます。同時分布の min-entropy は、揺らぎの目安（`expected_joint_min_entropy`）と比べます。
3. **連結したビット列**：C1 と同じ指標です。ビット数が $5N$ なので、揺らぎの目安も $5N$ で計算します。
-->

1. **Per qubit**: tables of each qubit's $\hat{P}_q(1)$ and bias and of the correlation coefficients between qubits. Compare the bias with the reference fluctuation $\sqrt{2/\pi}\,/\,(2\sqrt{N})$ and the correlation coefficients with $\pm 1/\sqrt{N}$.
2. **Joint distribution**: bar chart of $\hat{P}(x)$ for the 32 outcomes. The gray dotted lines are $1/32 \pm 2\sigma$; if most bars fall within this range, the deviations can be regarded as fluctuations. The min-entropy of the joint distribution is compared with the reference value for the fluctuation (`expected_joint_min_entropy`).
3. **Concatenated bit string**: the same metrics as C1. Since the number of bits is $5N$, the reference fluctuation is also computed with $5N$.

In [ ]:
r = evaluate_c2(counts_ideal)

print("===== (1) Per qubit =====")
show_per_qubit(r)

print()
print("===== (2) 5-bit joint distribution =====")
show_joint(r, title=f"Joint distribution (ideal, {GATE_IDEAL}, {r['shots']:,} shots)")

print()
print("===== (3) Concatenated bit string =====")
print(f"number of bits  = {r['n_bits']:,}")
show_metrics(r)

<!-- ### 4.5 測定を繰り返す -->

### 4.5 Repeating the measurement

<!-- 1 回の結果だけでは、偏りが回路のせいなのか偶然なのか分かりません。そこで同じ測定を `N_TRIALS` 回繰り返し、ばらつきを見ます。以降でも使うので、「回路を作る → transpile → 測定 → 評価」を関数 `run_c2` にまとめます。評価に必要な量はすべて `counts` から計算できるので、生のデータを保存しないときは `memory=True` を付けません。 -->

A single result cannot tell us whether a bias is due to the circuit or to chance. So we repeat the same measurement `N_TRIALS` times and look at the spread. Because it is also used later, we package "build the circuit → transpile → measure → evaluate" into the function `run_c2`. All quantities needed for evaluation can be computed from `counts`, so `memory=True` is not set when raw data is not being saved.

<!-- `run_c2` に `raw_path` を指定すると、連結したビット列を別ファイルに保存します。 -->

If `raw_path` is passed to `run_c2`, the concatenated bit string is saved to a separate file.

```python
r = run_c2("H", 200_000, seed=2026, raw_path="results/c2/manual/c2_H_ideal.txt")
```

<!--
`c2_H_ideal.txt` は、各 shot の 5 文字を Qiskit の文字列の順（量子ビット 4, 3, 2, 1, 0）のまま shot 順につなげた 0/1 の列（改行なし、長さ $5N$）です。
隣の `c2_H_ideal.txt.json` に測定条件・ノイズモデル・ビットの並び順・counts・評価結果を保存します。
既存ファイルがある場合は上書きせずエラーにします。保存先を省略すると、集計結果だけを返します。
繰り返し測定は `run_trials(..., raw_dir="results/c2/manual/trials")` で試行ごとに保存できます。
-->

`c2_H_ideal.txt` is a sequence of 0s and 1s (no line breaks, length $5N$) formed by joining the 5 characters of each shot in shot order, keeping Qiskit's string order (qubits 4, 3, 2, 1, 0).
The adjacent file `c2_H_ideal.txt.json` stores the measurement conditions, noise model, bit ordering, counts, and evaluation results.
If a file already exists, an error is raised rather than overwriting it. If the destination is omitted, only the aggregated results are returned.
For repeated measurements, `run_trials(..., raw_dir="results/c2/manual/trials")` saves the data for each trial.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、次のセルの最後の `run_trials` に `raw_dir` として `RUN_DIR/4.5_ideal_trials` を渡すので、試行ごとのファイル（`c2_H_ideal_trial001.txt` など）と、全試行の一覧 `trials.csv` を保存します。上の例のように自分で `raw_path` や `raw_dir` を指定した場合は、`SAVE_RESULTS` に関係なくその場所に保存します。 -->

If `SAVE_RESULTS = True` in 1., the `run_trials` call at the end of the next cell is given `RUN_DIR/4.5_ideal_trials` as `raw_dir`, so one file per trial (`c2_H_ideal_trial001.txt` and so on) and the list of all trials, `trials.csv`, are saved. If you give your own `raw_path` or `raw_dir` as in the examples above, the files are saved there regardless of `SAVE_RESULTS`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `run_c2` | `evaluate_c2` と同じ評価結果の辞書。保存時も戻り値は同じ | `run_c2("H", 200_000, seed=2026, raw_path="results/c2/manual/c2_H_ideal.txt")` |
| `run_trials` | `(試行番号, 評価結果の辞書)` のリスト。`raw_dir` を指定すると、試行ごとのファイルと `trials.csv` を保存する | `run_trials("H", 200_000, 3, seed=2026, raw_dir="results/c2/manual/trials")` |
| `run_conditions` | `(ラベル, 評価結果の辞書)` のリスト。`(ゲート, 条件の名前, NoiseModel)` の組を 1 回ずつ測定する（4.6・5.8 で使う）。`SAVE_RESULTS = True` なら raw data と `summary.csv` を保存する | `run_conditions([("H", "ideal", None)], 1_000, "4.6_ideal_gates")` |
| `offset_seed` | `seed + k`（`seed` が `None` なら `None`）。測定ごとに違うシードにする | `offset_seed(2026, 3)` |
| `summarize` | 連結したビット列の bias・min-entropy と、同時分布の min-entropy の要約を表示。戻り値は `None` | `summarize(rows_ideal)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `run_c2` | Evaluation dictionary, same as `evaluate_c2`. The return value is the same when saving | `run_c2("H", 200_000, seed=2026, raw_path="results/c2/manual/c2_H_ideal.txt")` |
| `run_trials` | List of `(trial number, evaluation dictionary)`. If `raw_dir` is given, one file per trial and `trials.csv` are saved | `run_trials("H", 200_000, 3, seed=2026, raw_dir="results/c2/manual/trials")` |
| `run_conditions` | List of `(label, evaluation dictionary)`. Measures each `(gate, condition name, NoiseModel)` once (used in 4.6 and 5.8). With `SAVE_RESULTS = True`, the raw data and `summary.csv` are saved | `run_conditions([("H", "ideal", None)], 1_000, "4.6_ideal_gates")` |
| `offset_seed` | `seed + k` (`None` if `seed` is `None`). Gives each measurement a different seed | `offset_seed(2026, 3)` |
| `summarize` | Displays a summary of the bias and min-entropy of the concatenated bit string and the min-entropy of the joint distribution. Returns `None` | `summarize(rows_ideal)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="8"><code>run_c2</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>測定回数（shot 数）。1 以上の整数（例：<code>200_000</code>）</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td><code>build_noise_model("both")</code> などで作った NoiseModel。ノイズなしなら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>再現用の非負整数（例：<code>2026</code>）。毎回異なる乱数なら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>raw_path</code></td><td>連結したビット列の保存先の文字列または Path（例：<code>"results/c2/manual/c2_H_ideal.txt"</code>）。<code>None</code> なら保存しない。未使用のファイル名を指定</td><td><code>None</code></td></tr>
<tr><td><code>method</code></td><td>通常は <code>"automatic"</code>。密度行列方式を指定する場合は <code>"density_matrix"</code></td><td><code>'automatic'</code></td></tr>
<tr><td><code>optimization_level</code></td><td>Qiskit の回路変換時の最適化レベル。<code>0</code>・<code>1</code>・<code>2</code>・<code>3</code>、既定値を使うなら <code>None</code>。5.9 では、最適化でゲートの順序や個数が変わらないよう <code>0</code> に固定</td><td><code>None</code></td></tr>
<tr><td><code>metadata</code></td><td>JSON に保存できる追加情報の辞書（例：<code>{"trial": 1}</code>）。不要なら <code>None</code>。<code>raw_path</code> 指定時に保存</td><td><code>None</code></td></tr>
<tr><td rowspan="7"><code>run_trials</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>1 試行あたりの測定回数。1 以上の整数</td><td>必須</td></tr>
<tr><td><code>n_trials</code></td><td>繰り返す試行数。1 以上の整数（例：<code>3</code>）</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td>NoiseModel。ノイズなしなら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>非負整数または <code>None</code>。整数の場合は試行番号を加えたシードで実行</td><td><code>None</code></td></tr>
<tr><td><code>raw_dir</code></td><td>試行別のビット列と JSON を保存するフォルダ。文字列または Path。<code>None</code> なら保存しない。既存ファイルは上書きしない</td><td><code>None</code></td></tr>
<tr><td><code>label</code></td><td>ファイル名に入れる条件の名前（例：<code>"ideal"</code>、<code>"noise-both"</code>）。<code>trials.csv</code> の <code>condition</code> 列にもなる</td><td><code>"ideal"</code></td></tr>
<tr><td rowspan="3"><code>run_conditions</code></td><td><code>conditions</code></td><td><code>(ゲート, 条件の名前, NoiseModel または None)</code> のリスト。条件の名前はファイル名に入る</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>1 回あたりの測定回数</td><td>必須</td></tr>
<tr><td><code>folder</code></td><td><code>RUN_DIR</code> の中の保存先のフォルダ名（例：<code>"4.6_ideal_gates"</code>）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>offset_seed</code></td><td><code>seed</code></td><td>非負整数または <code>None</code></td><td>必須</td></tr>
<tr><td><code>k</code></td><td>足す数（試行番号や条件の番号）</td><td>必須</td></tr>
<tr><td><code>summarize</code></td><td><code>rows</code></td><td><code>run_trials</code> が返す <code>(試行番号, 評価結果)</code> のリスト。少なくとも 1 試行が必要</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="8"><code>run_c2</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements (shots). An integer of at least 1 (e.g., <code>200_000</code>)</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td>A NoiseModel created with, e.g., <code>build_noise_model("both")</code>. <code>None</code> for no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Non-negative integer for reproducibility (e.g., <code>2026</code>). <code>None</code> for different random numbers each time</td><td><code>None</code></td></tr>
<tr><td><code>raw_path</code></td><td>Destination for the concatenated bit string, as a string or Path (e.g., <code>"results/c2/manual/c2_H_ideal.txt"</code>). <code>None</code> means do not save. Specify an unused file name</td><td><code>None</code></td></tr>
<tr><td><code>method</code></td><td>Normally <code>"automatic"</code>. Use <code>"density_matrix"</code> to select the density-matrix method</td><td><code>'automatic'</code></td></tr>
<tr><td><code>optimization_level</code></td><td>Optimization level for Qiskit's circuit conversion. <code>0</code>, <code>1</code>, <code>2</code>, <code>3</code>, or <code>None</code> to use the default. In 5.9 it is fixed at <code>0</code> so that optimization does not change the order or number of gates</td><td><code>None</code></td></tr>
<tr><td><code>metadata</code></td><td>Dictionary of extra information that can be saved as JSON (e.g., <code>{"trial": 1}</code>). <code>None</code> if not needed. Saved when <code>raw_path</code> is specified</td><td><code>None</code></td></tr>
<tr><td rowspan="7"><code>run_trials</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements per trial. An integer of at least 1</td><td>Required</td></tr>
<tr><td><code>n_trials</code></td><td>Number of trials to repeat. An integer of at least 1 (e.g., <code>3</code>)</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td>A NoiseModel. <code>None</code> for no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Non-negative integer or <code>None</code>. If an integer, each trial runs with the seed plus the trial number</td><td><code>None</code></td></tr>
<tr><td><code>raw_dir</code></td><td>Folder in which to save the bit string and JSON of each trial, as a string or Path. <code>None</code> means do not save. Existing files are not overwritten</td><td><code>None</code></td></tr>
<tr><td><code>label</code></td><td>Name of the condition put in the file names (e.g., <code>"ideal"</code>, <code>"noise-both"</code>). Also used as the <code>condition</code> column of <code>trials.csv</code></td><td><code>"ideal"</code></td></tr>
<tr><td rowspan="3"><code>run_conditions</code></td><td><code>conditions</code></td><td>List of <code>(gate, condition name, NoiseModel or None)</code>. The condition name goes into the file name</td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements per condition</td><td>Required</td></tr>
<tr><td><code>folder</code></td><td>Name of the destination folder in <code>RUN_DIR</code> (e.g., <code>"4.6_ideal_gates"</code>)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>offset_seed</code></td><td><code>seed</code></td><td>Non-negative integer or <code>None</code></td><td>Required</td></tr>
<tr><td><code>k</code></td><td>Number to add (trial number or condition number)</td><td>Required</td></tr>
<tr><td><code>summarize</code></td><td><code>rows</code></td><td>List of <code>(trial number, evaluation result)</code> returned by <code>run_trials</code>. At least one trial is required</td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- `run_c2` の `method`・`optimization_level`・`metadata` は、`method="density_matrix"` のように引数名を付けて指定します。保存する `.txt` は連結したビット列、`.txt.json` は条件と集計結果です。 -->

Specify `method`, `optimization_level`, and `metadata` of `run_c2` by keyword, as in `method="density_matrix"`. The saved `.txt` is the concatenated bit string, and the `.txt.json` holds the conditions and aggregated results.


In [ ]:
N_TRIALS = 10


def offset_seed(seed, k):
    # A different seed for each run: seed + k (None stays None, i.e., different results each time)
    return None if seed is None else seed + k


def run_c2(gate, shots, noise_model=None, seed=None, raw_path=None,
           *, method="automatic", optimization_level=None, metadata=None):
    # Build the circuit -> transpile -> measure -> evaluate. With raw_path, the raw data is also saved.
    if raw_path is not None:
        ensure_new(raw_path)   # stop before measuring if the files already exist
    qc_t = transpile(build_c2(gate), basis_gates=BASIS_GATES, optimization_level=optimization_level)
    backend = AerSimulator(method=method, noise_model=noise_model, seed_simulator=seed)
    result = backend.run(qc_t, shots=shots, memory=raw_path is not None).result()
    if raw_path is not None:
        save_raw(raw_path, result.get_memory(), gate, shots, seed, noise_model,
                 method=method, optimization_level=optimization_level, metadata=metadata)
    return evaluate_c2(result.get_counts())


def run_trials(gate, shots, n_trials, noise_model=None, seed=None, raw_dir=None, label="ideal"):
    # Repeat the same measurement. With raw_dir, one file per trial and trials.csv are saved.
    rows, records = [], []
    for trial in range(1, n_trials + 1):
        name = f"c2_{gate}_{label}_trial{trial:03d}.txt"
        raw_path = None if raw_dir is None else Path(raw_dir) / name
        s = offset_seed(seed, trial)
        r = run_c2(gate, shots, noise_model, s, raw_path, metadata={"condition": label, "trial": trial})
        rows.append((trial, r))
        records.append(dict(file=name, gate=gate, condition=label, trial=trial, seed=s, **r))
    if raw_dir is not None:
        write_csv(Path(raw_dir) / "trials.csv", records)   # written after all trials have finished
    return rows


def run_conditions(conditions, shots, folder):
    # Measure each (gate, condition, noise_model) once (used in 4.6 and 5.8).
    # With SAVE_RESULTS = True, the raw data and summary.csv are saved in output_dir(folder).
    out = output_dir(folder)
    rows, records = [], []
    for k, (gate, condition, noise_model) in enumerate(conditions):
        name = f"c2_{gate}_{condition}.txt"
        raw_path = None if out is None else out / name
        s = offset_seed(SEED, k)   # the same seed would align the results of different conditions
        r = run_c2(gate, shots, noise_model, s, raw_path, metadata={"condition": condition})
        rows.append((f"{gate} / {condition}", r))
        records.append(dict(file=name, gate=gate, condition=condition, seed=s, **r))
    if out is not None:
        write_csv(out / "summary.csv", records)
    return rows


def summarize(rows):
    mean = lambda xs: sum(xs) / len(xs)
    r0 = rows[0][1]
    biases = [r["bias"] for _, r in rows]
    mins = [r["min_entropy"] for _, r in rows]
    joints = [r["joint_min_entropy_per_bit"] for _, r in rows]
    print(f"[concatenated] mean bias        = {mean(biases):.6f} (reference {expected_bias(r0['n_bits']):.6f})")
    print(f"[concatenated] mean min-entropy = {mean(mins):.6f} (min {min(mins):.6f}, max {max(mins):.6f})")
    print(f"[joint] mean min-entropy / 5 = {mean(joints):.5f}"
          f" (reference {expected_joint_min_entropy(r0['shots']) / N_QUBITS:.5f})")


rows_ideal = run_trials(GATE_IDEAL, SHOTS_IDEAL, N_TRIALS, seed=SEED, raw_dir=output_dir("4.5_ideal_trials"))
show_table(rows_ideal, "trial")
summarize(rows_ideal)

<!-- ### 4.6 （参考）3 つのゲートを比べる -->

### 4.6 (Optional) Comparing the three gates

<!-- 同じ測定回数で $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ を 1 回ずつ測定して並べます。ノイズがないので、3 つの違いは統計的な揺らぎだけのはずです。 -->

We measure $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, and $R_Y(\pi/2)$ once each with the same number of measurements and list the results side by side. Since there is no noise, the only differences among the three should be statistical fluctuations.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、3 つのゲートの raw data と一覧 `summary.csv`を `4.6_ideal_gates/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data for the three gates and their list, `summary.csv`, are saved in `4.6_ideal_gates/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
rows = run_conditions([(gate, "ideal", None) for gate in GATES], SHOTS_IDEAL, "4.6_ideal_gates")
show_table(rows, "gate / condition")

<!-- ## 5. ノイズありのシミュレーション -->

## 5. Noisy simulation

<!-- 次に、説明パート E のノイズを加えて測定します。 -->

Next, we add the noise of Explanation Part E and measure.

<!-- ### 5.1 ノイズのパラメータ -->

### 5.1 Noise parameters

<!-- C1 のノートブックと同じ値を設定します（$\lambda$ は熱緩和と合わせた平均ゲート忠実度が 99.9% になる値です）。熱緩和誤差の関数には $T_1$, $T_2$, ゲート時間を**同じ単位**で渡す必要があるので、すべてナノ秒（ns）にそろえます。 -->

We set the same values as in the C1 notebook ($\lambda$ is the value that gives an average gate fidelity of 99.9% including thermal relaxation). The thermal relaxation error function requires $T_1$, $T_2$, and the gate time in the **same unit**, so everything is expressed in nanoseconds (ns).

In [ ]:
DEPOLARIZING_PARAM = 0.00198  # Depolarizing parameter λ (gives an average gate fidelity of 99.9% including thermal relaxation)
READOUT_ERROR_RATE = 0.03     # Readout error (fidelity 97%)

T1 = 0.964e6                  # Relaxation time 0.964 ms = 0.964e6 ns
T2 = 1.155e6                  # Dephasing time 1.155 ms = 1.155e6 ns
GATE_TIME = 20                # 1-qubit gate time 20 ns

print(f"Probability that |1> decays to |0> during the gate time ≈ {1 - math.exp(-GATE_TIME / T1):.2e}")

<!-- ### 5.2 ノイズモデルを作る関数 -->

### 5.2 Function for building the noise model

<!-- ノイズモデルは、ノイズの**成分**を組み合わせて作ります。次のセルの 3 つの部分でできています。 -->

The noise model is built by combining noise **components**. The next cell has three parts.

<!--
- `NOISE_COMPONENTS`：ノイズの成分の一覧。成分ごとに、強さから Qiskit Aer の誤差を作る関数を持ちます。ゲートの後に加える量子的な誤差は `gate_errors`、測定の読み出し誤差は `readout` に書きます。`gate_errors` のキーはゲートの名前（`"rx"`, `"ry"`）です。
- `NOISE_KINDS`：5.3〜5.8 で使うノイズの種類。名前と、`{成分: 強さ}` の組です。`"gate"` は脱分極 + 熱緩和、`"readout"` は対称な読み出し誤差、`"both"` はその両方で、強さは 5.1 の定数です。
- `build_noise_model(kind)`：`NOISE_KINDS` の名前、または `{成分: 強さ}` の辞書から `NoiseModel` を作ります。同じゲートの誤差は書いた順に `compose` で合成し（脱分極のあとに熱緩和）、読み出し誤差は混同行列を掛け合わせて、それぞれ 1 回だけ加えます。Qiskit Aer では、読み出し誤差を 2 回加えると後のもので上書きされるためです。5.9 の `build_sweep_noise` も同じ関数を使います。
-->

- `NOISE_COMPONENTS`: the list of noise components. Each component has functions that create Qiskit Aer errors from a strength. Quantum errors added after gates go in `gate_errors`, and readout errors at measurement go in `readout`. The keys of `gate_errors` are gate names (`"rx"`, `"ry"`).
- `NOISE_KINDS`: the noise kinds used in 5.3–5.8. Each is a name and a set of `{component: strength}`. `"gate"` is depolarizing + thermal relaxation, `"readout"` is a symmetric readout error, and `"both"` is both; the strengths are the constants in 5.1.
- `build_noise_model(kind)`: builds a `NoiseModel` from a name in `NOISE_KINDS` or from a dictionary `{component: strength}`. Errors on the same gate are composed with `compose` in the order written (depolarizing, then thermal relaxation), readout errors are combined by multiplying their confusion matrices, and each is added only once, because in Qiskit Aer adding a second readout error replaces the first. `build_sweep_noise` in 5.9 uses the same function.

<!-- このノートブックで定義している成分は次のとおりです。 -->

The components defined in this notebook are as follows.

<!--
| 成分 | 加える誤差 | 強さ |
|---|---|---|
| `depolarizing` | `rx`, `ry` の後に `depolarizing_error(λ, 1)` | λ（0〜1） |
| `thermal` | `rx`, `ry` の後に `thermal_relaxation_error(T1, T2, 倍率 × GATE_TIME)` | ゲート時間の倍率（1 が 5.1 の値） |
| `readout` | 対称な読み出し誤差の混同行列 $\begin{pmatrix} 1-\varepsilon & \varepsilon \\ \varepsilon & 1-\varepsilon \end{pmatrix}$ | ε（0〜1） |
-->

| Component | Error added | Strength |
|---|---|---|
| `depolarizing` | `depolarizing_error(λ, 1)` after `rx`, `ry` | λ (0–1) |
| `thermal` | `thermal_relaxation_error(T1, T2, multiplier × GATE_TIME)` after `rx`, `ry` | Gate-time multiplier (1 is the value in 5.1) |
| `readout` | Confusion matrix of a symmetric readout error, $\begin{pmatrix} 1-\varepsilon & \varepsilon \\ \varepsilon & 1-\varepsilon \end{pmatrix}$ | ε (0–1) |

<!-- **ノイズの種類を増やす手順** -->

**How to add a kind of noise**

<!--
1. `NOISE_COMPONENTS` に成分を 1 つ加えます（下の例と、その下の Qiskit Aer の誤差の表を参照）。
2. 5.3〜5.8 で使うときは、`NOISE_KINDS` に名前と `{成分: 強さ}` を加え、5.3 の `NOISE_KIND` にその名前を指定します。5.8 の比較と、保存するファイルの名前（`noise-<名前>`）は自動で増えます。
3. 5.9 で強さを変えるときは、5.9 の `SWEEP_VALUES` に成分の名前と強さのリストを加えます。グラフの列、CSV、保存のフォルダは自動で増えます。
-->

1. Add one component to `NOISE_COMPONENTS` (see the example below and the table of Qiskit Aer errors after it).
2. To use it in 5.3–5.8, add a name and `{component: strength}` to `NOISE_KINDS`, and set `NOISE_KIND` in 5.3 to that name. The comparison in 5.8 and the names of the saved files (`noise-<name>`) follow automatically.
3. To vary its strength in 5.9, add the component name and a list of strengths to `SWEEP_VALUES` in 5.9. The plot columns, the CSV files, and the saved folders follow automatically.

<!-- 例：エネルギー緩和だけを表す振幅減衰を加える場合（次のセルの後に、新しいセルで実行します） -->

Example: adding amplitude damping, which models energy relaxation only (run in a new cell after the next cell)

```python
from qiskit_aer.noise import amplitude_damping_error

NOISE_COMPONENTS["amplitude_damping"] = dict(
    label="Amplitude damping probability gamma", max=1,
    gate_errors=lambda s: {("rx", "ry"): amplitude_damping_error(s)})
NOISE_KINDS["amp_damping"] = {"amplitude_damping": 0.01}   # NOISE_KIND = "amp_damping" in 5.3
# In the settings cell of 5.9: SWEEP_VALUES["amplitude_damping"] = [i / 50 for i in range(11)]
```

<!-- **Qiskit Aer で加えられるノイズ**（qiskit-aer 0.17） -->

**Noise that Qiskit Aer can add** (qiskit-aer 0.17)

<!-- 量子的な誤差（`QuantumError`）はゲートの後に、古典的な読み出し誤差（`ReadoutError`）は測定のときに加わります。量子的な誤差は `compose`（順に作用）や `tensor`（量子ビットを並べる）で組み合わせられます。 -->

Quantum errors (`QuantumError`) act after gates, and classical readout errors (`ReadoutError`) act at measurement. Quantum errors can be combined with `compose` (applied in sequence) and `tensor` (placed on qubits side by side).

<!--
| 関数（`qiskit_aer.noise`） | 種類 | 主な引数 | 内容・使いどころ |
|---|---|---|---|
| `depolarizing_error(param, num_qubits)` | 量子（ゲートの後） | 脱分極パラメータ λ、量子ビット数 | 確率 λ で状態を最大混合状態に置き換える。ゲートの不忠実度をまとめて表す標準的なモデル |
| `thermal_relaxation_error(t1, t2, time, excited_state_population=0)` | 量子 | $T_1$, $T_2$, 時間 | エネルギー緩和（$\|1\rangle \to \|0\rangle$）と位相緩和。ゲート時間や待ち時間の間の緩和 |
| `amplitude_damping_error(param_amp, excited_state_population=0)` | 量子 | 減衰確率 γ | エネルギー緩和だけ（$\|1\rangle \to \|0\rangle$） |
| `phase_damping_error(param_phase)` | 量子 | 位相減衰の強さ | 位相緩和だけ。重ね合わせの干渉が弱まるが、計算基底での測定確率は変えない |
| `phase_amplitude_damping_error(param_amp, param_phase, excited_state_population=0)` | 量子 | γ と位相減衰の強さ | 上の 2 つを同時に |
| `pauli_error([("X", p), ("I", 1 - p)])` | 量子 | パウリ演算子と確率の組 | ビット反転（`X`）、位相反転（`Z`）、両方（`Y`）など。`"XX"` のように 2 量子ビットの相関した誤差も書ける |
| `coherent_unitary_error(unitary)` | 量子 | ユニタリ行列 | 確率的でない系統誤差。回転角のずれ（過回転）など。例：`RXGate(0.01).to_matrix()` |
| `mixed_unitary_error([(U1, p1), (U2, p2), ...])` | 量子 | ユニタリ行列と確率の組 | 確率で選ばれるユニタリ。パウリ以外の確率的な誤差 |
| `reset_error(prob0, prob1=0)` | 量子 | 確率 | 確率で $\|0\rangle$（または $\|1\rangle$）にリセットされる。状態準備の誤りなど |
| `kraus_error(noise_ops)` | 量子 | Kraus 演算子のリスト | 任意の量子チャネル。上にない誤差を自分で定義する |
| `ReadoutError(probabilities)` | 古典（測定時） | 混同行列 | 測定値の読み間違い。非対称（$P(1\mid 0) \ne P(0\mid 1)$）も表せる |
-->

| Function (`qiskit_aer.noise`) | Type | Main arguments | What it models |
|---|---|---|---|
| `depolarizing_error(param, num_qubits)` | Quantum (after gates) | Depolarizing parameter λ, number of qubits | Replaces the state with the maximally mixed state with probability λ. The standard model summarizing gate infidelity |
| `thermal_relaxation_error(t1, t2, time, excited_state_population=0)` | Quantum | $T_1$, $T_2$, time | Energy relaxation ($\|1\rangle \to \|0\rangle$) and dephasing during a gate time or idle time |
| `amplitude_damping_error(param_amp, excited_state_population=0)` | Quantum | Damping probability γ | Energy relaxation only ($\|1\rangle \to \|0\rangle$) |
| `phase_damping_error(param_phase)` | Quantum | Dephasing strength | Dephasing only. Weakens the interference of superpositions but does not change the measurement probabilities in the computational basis |
| `phase_amplitude_damping_error(param_amp, param_phase, excited_state_population=0)` | Quantum | γ and the dephasing strength | Both of the above at once |
| `pauli_error([("X", p), ("I", 1 - p)])` | Quantum | Pairs of Pauli operator and probability | Bit flip (`X`), phase flip (`Z`), both (`Y`), and so on. Correlated 2-qubit errors such as `"XX"` can also be written |
| `coherent_unitary_error(unitary)` | Quantum | Unitary matrix | A non-random systematic error, such as an over-rotation. Example: `RXGate(0.01).to_matrix()` |
| `mixed_unitary_error([(U1, p1), (U2, p2), ...])` | Quantum | Pairs of unitary and probability | A unitary chosen at random. Random errors other than Paulis |
| `reset_error(prob0, prob1=0)` | Quantum | Probabilities | Resets to $\|0\rangle$ (or $\|1\rangle$) with some probability, e.g., state-preparation errors |
| `kraus_error(noise_ops)` | Quantum | List of Kraus operators | Any quantum channel. For defining an error not listed above |
| `ReadoutError(probabilities)` | Classical (at measurement) | Confusion matrix | Misreading of the measured value. Can be asymmetric ($P(1\mid 0) \ne P(0\mid 1)$) |

<!--
| `NoiseModel` のメソッド | 内容 |
|---|---|
| `add_all_qubit_quantum_error(error, instructions)` | すべての量子ビットの、指定したゲートの後に誤差を加える（このノートブックの `build_noise_model` の方法） |
| `add_quantum_error(error, instructions, qubits)` | 特定の量子ビット（の組）だけに誤差を加える。量子ビットごとに違う誤差や、特定の組の相関した誤差に使う |
| `add_all_qubit_readout_error(error)` / `add_readout_error(error, qubits)` | すべての / 特定の量子ビットの測定に読み出し誤差を加える |
| `NoiseModel.from_backend(backend)` | 実機（または実機を模した backend）のキャリブレーションの値からノイズモデルを作る |
-->

| `NoiseModel` method | What it does |
|---|---|
| `add_all_qubit_quantum_error(error, instructions)` | Adds an error after the specified gates on all qubits (the method used by `build_noise_model` in this notebook) |
| `add_quantum_error(error, instructions, qubits)` | Adds an error only on specific qubits (or pairs of qubits). For errors that differ by qubit, or correlated errors on a specific pair |
| `add_all_qubit_readout_error(error)` / `add_readout_error(error, qubits)` | Adds a readout error to the measurement of all / specific qubits |
| `NoiseModel.from_backend(backend)` | Builds a noise model from the calibration values of a real device (or a backend that mimics one) |

<!--
`NoiseModel` では表せないものもあります。

- **shot どうしの相関**：各 shot は独立にシミュレーションされるので、前の測定結果が次の結果に影響するような相関は作れません。測定後のビット列を加工する必要があります。
- **時間とともに変わるノイズ（ドリフト）**：1 回の実行の間、ノイズモデルは変わりません。測定を分けて、強さを変えながら実行する必要があります。
- **量子ビットの外への漏れ（リーク）**：量子ビットの 2 準位の外の状態は扱えません。
-->

Some things cannot be expressed with `NoiseModel`.

- **Correlations between shots**: each shot is simulated independently, so correlations in which one measurement result affects the next cannot be created. The measured bit string has to be processed afterwards.
- **Noise that changes over time (drift)**: the noise model does not change during one run. The measurement has to be split into parts run with different strengths.
- **Leakage out of the qubit**: states outside the two levels of the qubit cannot be represented.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_noise_model` | ゲートの誤差と読み出し誤差を加えた NoiseModel | `build_noise_model("both")`、`build_noise_model({"thermal": 1000})` |
| `noise_components` | 確かめた `{成分: 強さ}` の辞書。知らない成分や範囲外の強さならエラー | `noise_components("both")` |
| `readout_confusion` | 読み出しの成分の混同行列を掛け合わせた 2×2 の numpy 配列。読み出しの成分がなければ `None` | `readout_confusion({"readout": 0.03})` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_noise_model` | NoiseModel with gate errors and readout errors | `build_noise_model("both")`, `build_noise_model({"thermal": 1000})` |
| `noise_components` | The checked dictionary `{component: strength}`. An error is raised for an unknown component or a strength out of range | `noise_components("both")` |
| `readout_confusion` | 2×2 numpy array: the product of the confusion matrices of the readout components. `None` if there are no readout components | `readout_confusion({"readout": 0.03})` |

<br>

<!--
| 関数 | 引数 | 入力する値 | 省略時 |
|---|---|---|---|
| `build_noise_model`, `noise_components`, `readout_confusion` | `kind` | `NOISE_KINDS` の名前（`"gate"`, `"readout"`, `"both"` など）、または `{成分の名前: 強さ}` の辞書。強さは 0 以上で、成分の `max` 以下 | 必須 |
-->

| Function | Argument | Value to pass | If omitted |
|---|---|---|---|
| `build_noise_model`, `noise_components`, `readout_confusion` | `kind` | A name in `NOISE_KINDS` (`"gate"`, `"readout"`, `"both"`, and so on), or a dictionary `{component name: strength}`. Each strength must be at least 0 and at most the component's `max` | Required |

<br>

<!--
`NOISE_COMPONENTS` の各成分に書く項目は次のとおりです。

| 項目 | 内容 | 省略時 |
|---|---|---|
| `label` | 強さの意味。5.9 のグラフの横軸の名前にもなる | 必須 |
| `max` | 強さの上限。`None` なら上限なし | `None` |
| `gate_errors` | 強さ → `{(ゲートの名前, ...): QuantumError}` の関数。誤差をすべての量子ビットのそのゲートの後に加える | なし |
| `readout` | 強さ → 混同行列 `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]` の関数。すべての量子ビットの測定に加える | なし |
| `log_scale` | `True` なら、5.9 のグラフの横軸を、0〜1 は線形、1 以上は対数の目盛り（symlog）にする | `False` |
-->

The items written for each component in `NOISE_COMPONENTS` are as follows.

| Item | Contents | If omitted |
|---|---|---|
| `label` | What the strength means. Also used as the horizontal-axis label in 5.9 | Required |
| `max` | Upper limit of the strength. `None` means no upper limit | `None` |
| `gate_errors` | Function strength → `{(gate name, ...): QuantumError}`. The error is added after those gates on all qubits | None |
| `readout` | Function strength → confusion matrix `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]`. Added to the measurement of all qubits | None |
| `log_scale` | If `True`, the horizontal axis in 5.9 is linear from 0 to 1 and logarithmic above 1 (symlog) | `False` |

In [ ]:
# ---- Noise components ----
# Each component is one kind of noise whose strength can be set. To add a new kind of noise,
# add an entry here (the table in 5.2 lists the errors Qiskit Aer provides).
#   label       : what the strength means (also the horizontal-axis label in 5.9)
#   max         : largest allowed strength (None: no upper limit)
#   gate_errors : function strength -> {(gate names, ...): QuantumError}; the error is added after those gates on all qubits
#   readout     : function strength -> confusion matrix [[P(0|0), P(1|0)], [P(0|1), P(1|1)]] for all measurements
#   log_scale   : True to draw the horizontal axis in 5.9 on a symlog scale
NOISE_COMPONENTS = {
    "depolarizing": dict(
        label="Depolarizing parameter lambda", max=1,
        gate_errors=lambda s: {("rx", "ry"): depolarizing_error(s, 1)}),
    "thermal": dict(
        label="Thermal gate-time multiplier", max=None, log_scale=True,
        gate_errors=lambda s: {("rx", "ry"): thermal_relaxation_error(T1, T2, s * GATE_TIME)}),
    "readout": dict(
        label="Symmetric readout error epsilon", max=1,
        readout=lambda s: [[1 - s, s], [s, 1 - s]]),
}

# ---- Noise kinds used in 5.3-5.8: name -> {component: strength} ----
# Components are applied in the order written (here: depolarizing, then thermal relaxation).
# The strength of "thermal" is the multiplier of GATE_TIME (1 = the gate time in 5.1).
NOISE_KINDS = {
    "gate": {"depolarizing": DEPOLARIZING_PARAM, "thermal": 1},
    "readout": {"readout": READOUT_ERROR_RATE},
    "both": {"depolarizing": DEPOLARIZING_PARAM, "thermal": 1, "readout": READOUT_ERROR_RATE},
}


def noise_components(kind):
    # A name in NOISE_KINDS, or a dictionary {component: strength} -> the checked dictionary {component: strength}
    components = NOISE_KINDS.get(kind) if isinstance(kind, str) else kind
    if components is None:
        raise ValueError(f"kind must be one of {list(NOISE_KINDS)} or a dictionary {{component: strength}} (got: {kind!r})")
    for name, strength in components.items():
        if name not in NOISE_COMPONENTS:
            raise ValueError(f"Unknown noise component: {name!r} (defined: {list(NOISE_COMPONENTS)})")
        upper = NOISE_COMPONENTS[name].get("max")
        if not math.isfinite(strength) or strength < 0 or (upper is not None and strength > upper):
            raise ValueError(f"The strength of {name!r} must be between 0 and {upper} (got: {strength})")
    return components


def readout_confusion(kind):
    # Product of the confusion matrices of the readout components (None if there are none)
    confusion = None
    for name, strength in noise_components(kind).items():
        make = NOISE_COMPONENTS[name].get("readout")
        if make is None or strength == 0:
            continue
        matrix = np.array(make(strength), dtype=float)
        confusion = matrix if confusion is None else confusion @ matrix
    return confusion


def build_noise_model(kind):
    noise_model = NoiseModel(basis_gates=BASIS_GATES)
    # Errors on the same gate are composed in the order of the components and added once
    gate_errors = {}
    for name, strength in noise_components(kind).items():
        make = NOISE_COMPONENTS[name].get("gate_errors")
        if make is None or strength == 0:
            continue
        for gates, error in make(strength).items():
            for gate in gates:
                gate_errors[gate] = error if gate not in gate_errors else gate_errors[gate].compose(error)
    for gate, error in gate_errors.items():
        noise_model.add_all_qubit_quantum_error(error, [gate])
    # Readout errors are combined into one confusion matrix (adding a second ReadoutError would replace the first)
    confusion = readout_confusion(kind)
    if confusion is not None:
        noise_model.add_all_qubit_readout_error(ReadoutError(confusion))
    return noise_model


for kind in NOISE_KINDS:
    print(f"===== {kind} =====")
    print(build_noise_model(kind))

<!-- ### 5.3 ゲート・ノイズの種類・測定回数を選ぶ -->

### 5.3 Choosing the gate, noise type, and number of measurements

<!-- `GATE_NOISY` に `"H"`, `"Rx"`, `"Ry"` のいずれかを、`NOISE_KIND` に `"gate"`, `"readout"`, `"both"` のいずれか（または 5.2 の `NOISE_KINDS` に自分で加えた名前）を指定します。測定回数は 4. と同じ $2 \times 10^5$ shots（連結して $10^6$ ビット）です。C1 と同じく、この回数ではゲートノイズによる偏り（約 $10^{-5}$）は統計的な揺らぎ（連結したビット列で約 $4 \times 10^{-4}$）に埋もれて検出できません。ここで確かめるのは、3 つの見方のどれでも、結果が揺らぎの範囲に収まることです。 -->

Set `GATE_NOISY` to one of `"H"`, `"Rx"`, `"Ry"`, and set `NOISE_KIND` to one of `"gate"`, `"readout"`, `"both"` (or a name you added to `NOISE_KINDS` in 5.2). The number of measurements is $2 \times 10^5$ shots ($10^6$ bits when concatenated), the same as in Section 4. As in C1, at this number the bias from gate noise (about $10^{-5}$) is buried in the statistical fluctuation (about $4 \times 10^{-4}$ for the concatenated bit string) and cannot be detected. What we check here is that the results stay within the range of the fluctuation from all three viewpoints.

In [ ]:
GATE_NOISY = "H"          # Choose from "H", "Rx", "Ry"
NOISE_KIND = "both"       # A name in NOISE_KINDS (5.2): "gate", "readout", "both", ...
SHOTS_NOISY = 200_000     # Number of measurements

noise_model = build_noise_model(NOISE_KIND)

<!-- ### 5.4 回路を作って transpile する -->

### 5.4 Building and transpiling the circuit

<!-- 4.2 と同じです。ノイズはゲート `rx`, `ry` に加えるので、transpile 後のゲートの数がそのまま「ノイズを受ける回数」になります（5 量子ビットの合計で、$H_{\mathrm{Gate}}$ は 10 回、$R_X$, $R_Y$ は 5 回）。 -->

This is the same as 4.2. Since noise is added at the `rx` and `ry` gates, the number of gates after transpilation is exactly the "number of times noise is received" (over the 5 qubits in total, 10 for $H_{\mathrm{Gate}}$ and 5 for $R_X$ and $R_Y$).

In [ ]:
qc_noisy = build_c2(GATE_NOISY)
qc_noisy_t = transpile(qc_noisy, basis_gates=BASIS_GATES)

print(qc_noisy_t.draw())
print("Number of gates receiving noise (total over 5 qubits):",
      sum(qc_noisy_t.count_ops().get(g, 0) for g in ("rx", "ry")))

<!-- ### 5.5 測定する -->

### 5.5 Measuring

<!-- 4.3 との違いは、`AerSimulator(noise_model=noise_model)` でノイズモデルを渡すところだけです。 -->

The only difference from 4.3 is that the noise model is passed in via `AerSimulator(noise_model=noise_model)`.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、このセルの raw dataを `5.5_noisy_single/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data of this cell is saved in `5.5_noisy_single/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
backend_noisy = AerSimulator(noise_model=noise_model, seed_simulator=SEED)

job = backend_noisy.run(qc_noisy_t, shots=SHOTS_NOISY, memory=True)
result = job.result()

counts_noisy = result.get_counts()
raw_shots_noisy = result.get_memory()
flat_bits_noisy = "".join(raw_shots_noisy)

print("Number of distinct outcomes =", len(counts_noisy))
print("First 5 shots =", raw_shots_noisy[:5])
print("Concatenated bit string (first 50 bits) =", flat_bits_noisy[:50])
print("Length of the concatenated bit string =", len(flat_bits_noisy))

# Save the raw data (only when SAVE_RESULTS = True)
if SAVE_RESULTS:
    path = save_raw(output_dir("5.5_noisy_single") / f"c2_{GATE_NOISY}_noise-{NOISE_KIND}.txt", raw_shots_noisy,
                    GATE_NOISY, SHOTS_NOISY, SEED, noise_model, metadata={"condition": f"noise-{NOISE_KIND}"})
    print("Saved:", path)

<!-- ### 5.6 評価指標を計算する -->

### 5.6 Computing the evaluation metrics

<!-- 4.4 と同じ指標を計算します。説明パート E の予想どおり、3 つの見方のどれでも、結果が統計的な揺らぎの目安と同程度かどうかを確認してください。 -->

We compute the same metrics as in 4.4. Check whether, as predicted in Explanation Part E, the results from all three viewpoints are comparable to the reference values for statistical fluctuation.

In [ ]:
r = evaluate_c2(counts_noisy)

print("===== (1) Per qubit =====")
show_per_qubit(r)

print()
print("===== (2) 5-bit joint distribution =====")
show_joint(r, title=f"Joint distribution (noisy: {NOISE_KIND}, {GATE_NOISY}, {r['shots']:,} shots)")

print()
print("===== (3) Concatenated bit string =====")
show_metrics(r)

<!-- ### 5.7 測定を繰り返す -->

### 5.7 Repeating the measurement

<!-- 4.5 の `run_trials` にノイズモデルを渡して、`N_TRIALS` 回繰り返します。 -->

We pass the noise model to `run_trials` from 4.5 and repeat `N_TRIALS` times.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、試行ごとのファイル（`c2_H_noise-both_trial001.txt` など）と `trials.csv`を `5.7_noisy_trials/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., one file per trial (`c2_H_noise-both_trial001.txt` and so on) and `trials.csv` are saved in `5.7_noisy_trials/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
rows_noisy = run_trials(GATE_NOISY, SHOTS_NOISY, N_TRIALS, noise_model=noise_model, seed=SEED,
                        raw_dir=output_dir("5.7_noisy_trials"), label=f"noise-{NOISE_KIND}")
show_table(rows_noisy, "trial")
summarize(rows_noisy)

<!-- ### 5.8 （参考）ゲートとノイズの種類をすべて比べる -->

### 5.8 (Optional) Comparing all gates and noise types

<!-- 3 つのゲート × 3 種類のノイズの 9 通りを 1 回ずつ測定して並べます（$2 \times 10^5$ shots × 9 回）。 -->

We measure all 9 combinations of 3 gates × 3 noise types once each and list the results side by side ($2 \times 10^5$ shots × 9 runs).

<!-- 1. で `SAVE_RESULTS = True` にした場合は、9 通りの raw data（`c2_H_noise-gate.txt` など）と一覧 `summary.csv`を `5.8_noisy_comparison/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data for the 9 combinations (`c2_H_noise-gate.txt` and so on) and their list, `summary.csv`, are saved in `5.8_noisy_comparison/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
conditions = [(gate, f"noise-{kind}", build_noise_model(kind)) for kind in NOISE_KINDS for gate in GATES]
rows = run_conditions(conditions, SHOTS_NOISY, "5.8_noisy_comparison")
show_table(rows, "gate / noise")
print(f"[Reference] bias of the concatenated bit string = {expected_bias(5 * SHOTS_NOISY):.6f}, "
      f"min-entropy / 5 of the joint distribution = {expected_joint_min_entropy(SHOTS_NOISY) / N_QUBITS:.5f}")

<!-- ### 5.9 ノイズの大きさを段階的に変える -->

### 5.9 Changing the noise strength step by step

<!-- C1 のノートブックの 5.9 と同じく、脱分極・熱緩和・読み出し誤差を**1 成分ずつ**変更し、ほかの成分をゼロにして原因を切り分けます。各条件で $H_{\mathrm{Gate}}$ / Rx / Ry を比較します。 -->

As in 5.9 of the C1 notebook, we vary the depolarizing, thermal relaxation, and readout errors **one component at a time**, setting the other components to zero to isolate the cause. Under each condition we compare $H_{\mathrm{Gate}}$ / Rx / Ry.

<!--
- 脱分極：`depolarizing_error` のパラメータ λ を 0〜0.10 で変えます（平均ゲート不忠実度そのものではありません）。
- 熱緩和：T1、T2 は 5.1 の値に固定し、ノイズに渡す有効ゲート時間を `倍率 × GATE_TIME` にします。倍率 0 は熱緩和なし、1 は元の条件です。大きい倍率は影響を観察するための仮想条件です。
- 読み出し：対称な読み出し誤差率 $\varepsilon$ を 0〜0.20 で変えます。
-->

- Depolarizing: vary the parameter λ of `depolarizing_error` from 0 to 0.10 (this is not the average gate infidelity itself).
- Thermal relaxation: fix T1 and T2 at the values in 5.1, and set the effective gate time passed to the noise to `multiplier × GATE_TIME`. A multiplier of 0 means no thermal relaxation, and 1 is the original condition. Large multipliers are hypothetical conditions for observing the effect.
- Readout: vary the symmetric readout error rate $\varepsilon$ from 0 to 0.20.

<!-- 縦軸は、連結したビット列の $P(1)$・bias と、同時分布の min-entropy（1 ビットあたり）の 3 つです。C2 回路では 5 個の量子ビットに同じノイズが独立に加わるので、連結したビット列の $P(1)$ は C1 回路の $P(1)$ と同じになるはずです。C1 のノートブックの 5.9 のグラフと比べてみてください。 -->

The vertical axes are three quantities: $P(1)$ and bias of the concatenated bit string, and the min-entropy (per bit) of the joint distribution. In the C2 circuit, the same noise is applied independently to the 5 qubits, so $P(1)$ of the concatenated bit string should equal $P(1)$ of the C1 circuit. Try comparing with the plots in 5.9 of the C1 notebook.

<!-- `SWEEP_VALUES` のリストを編集すると、範囲と刻み幅を自由に変更できます。$2 \times 10^5$ shots（連結して $10^6$ ビット）× 3 試行で確認します。全条件数と総 shots は実行前に表示します。 -->

The range and step size can be changed freely by editing the lists in `SWEEP_VALUES`. We check with $2 \times 10^5$ shots ($10^6$ bits when concatenated) × 3 trials. The total number of conditions and total shots are displayed before running.

<!-- ノイズの成分は 5.2 の `NOISE_COMPONENTS` で定義しています。新しい成分を 5.2 で加えたあと、`SWEEP_VALUES` に成分の名前と強さのリストを加えると、その成分も強さを変えて測定します。グラフの列は成分の数だけ自動で増えます。 -->

The noise components are defined in `NOISE_COMPONENTS` in 5.2. After adding a new component in 5.2, add its name and a list of strengths to `SWEEP_VALUES` to vary its strength as well. The plot gets one column per component automatically.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_sweep_noise` | 指定した 1 成分だけを含む NoiseModel | `build_sweep_noise("thermal", 1000)` |
| `sweep_exact` | 測定揺らぎを含まない、モデルから計算した指標の辞書（`P1`・`bias`・`min_entropy`・`joint_min_entropy_per_bit`） | `sweep_exact("H", "thermal", 1000, build_sweep_noise("thermal", 1000))` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_sweep_noise` | NoiseModel containing only the specified single component | `build_sweep_noise("thermal", 1000)` |
| `sweep_exact` | Dictionary of metrics computed from the model, without measurement fluctuation (`P1`, `bias`, `min_entropy`, `joint_min_entropy_per_bit`) | `sweep_exact("H", "thermal", 1000, build_sweep_noise("thermal", 1000))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>build_sweep_noise</code></td><td><code>component</code></td><td>5.2 の <code>NOISE_COMPONENTS</code> にある成分の名前で、<code>SWEEP_VALUES</code> のキーにもあるもの（例：<code>"thermal"</code>）</td><td>必須</td></tr>
<tr><td><code>strength</code></td><td>有限の非負値。脱分極は λ（0〜1）、読み出しは誤り確率 ε（0〜1）、熱緩和は <code>GATE_TIME</code> の倍率（例：<code>1000</code>）。<code>0</code> はノイズなし</td><td>必須</td></tr>
<tr><td rowspan="4"><code>sweep_exact</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>component</code></td><td><code>build_sweep_noise</code> と同じノイズ成分名</td><td>必須</td></tr>
<tr><td><code>strength</code></td><td><code>build_sweep_noise</code> に渡したものと同じ強さ。読み出し誤差を確率に反映するときに使用</td><td>必須</td></tr>
<tr><td><code>nm</code></td><td>同じ <code>component</code>・<code>strength</code> で <code>build_sweep_noise</code> が返した NoiseModel</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>build_sweep_noise</code></td><td><code>component</code></td><td>A component name in <code>NOISE_COMPONENTS</code> (5.2) that is also a key of <code>SWEEP_VALUES</code> (e.g., <code>"thermal"</code>)</td><td>Required</td></tr>
<tr><td><code>strength</code></td><td>A finite non-negative value. λ (0–1) for depolarizing, error probability ε (0–1) for readout, multiplier of <code>GATE_TIME</code> (e.g., <code>1000</code>) for thermal relaxation. <code>0</code> means no noise</td><td>Required</td></tr>
<tr><td rowspan="4"><code>sweep_exact</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>component</code></td><td>Same noise component name as for <code>build_sweep_noise</code></td><td>Required</td></tr>
<tr><td><code>strength</code></td><td>Same strength as passed to <code>build_sweep_noise</code>. Used to reflect the readout error in the probabilities</td><td>Required</td></tr>
<tr><td><code>nm</code></td><td>NoiseModel returned by <code>build_sweep_noise</code> for the same <code>component</code> and <code>strength</code></td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- `sweep_exact` は、測定直前の密度行列の対角成分から 32 通りの確率を求め、読み出し誤差があるときは各量子ビットに混同行列をかけて、測定される確率を計算します。 -->

`sweep_exact` obtains the probabilities of the 32 outcomes from the diagonal elements of the density matrix just before measurement and, if there is readout error, applies the confusion matrix to each qubit to compute the probabilities that are measured.


In [ ]:
SWEEP_VALUES = {
    "depolarizing": [i / 100 for i in range(11)],
    "thermal": [0, 1, 10, 100, 250, 500, 1000, 2500, 5000],
    "readout": [i / 50 for i in range(11)],
}
SWEEP_GATES = list(GATES)
SWEEP_SHOTS = 200_000
SWEEP_TRIALS = 3
SWEEP_SEED = 2026


def build_sweep_noise(component, strength):
    # NoiseModel with only one component (all other components are zero)
    if component not in SWEEP_VALUES:
        raise ValueError(f"Unknown noise component: {component} (add it to SWEEP_VALUES)")
    return build_noise_model({component: strength})


def apply_readout(probs, confusion):
    # probs: probabilities of length 32 (the binary expansion of index k is q4 ... q0). Apply the confusion matrix to each qubit
    p = np.asarray(probs, dtype=float).reshape((2,) * N_QUBITS)
    for axis in range(N_QUBITS):
        p = np.moveaxis(np.tensordot(p, confusion, axes=([axis], [0])), -1, axis)
    return p.reshape(-1)


def sweep_exact(gate, component, strength, nm):
    # Save the density matrix before measurement. Readout error is applied with a classical confusion matrix.
    qc = transpile(build_c2(gate), basis_gates=BASIS_GATES, optimization_level=0)
    qc = qc.remove_final_measurements(inplace=False)
    qc.save_density_matrix()
    backend = AerSimulator(method="density_matrix", noise_model=nm)
    rho = backend.run(qc, shots=1).result().data(0)["density_matrix"]
    probs = np.clip(np.real(np.diag(rho.data)), 0, 1)
    confusion = readout_confusion({component: strength})
    if confusion is not None:
        probs = apply_readout(probs, confusion)
    probs = probs / probs.sum()
    counts = {x: float(p) for x, p in zip(ALL_BITSTRINGS, probs)}   # Pass probabilities as 'counts'
    r = evaluate_c2(counts)
    return {key: r[key] for key in ("P1", "bias", "min_entropy", "joint_min_entropy_per_bit")}

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `run_noise_sweep` | `(試行ごとの記録リスト, 条件ごとの要約リスト)` のタプル。`raw_dir` を指定すると、raw data と `trials.csv`・`summary.csv` をそのフォルダに保存する | `records, summary = run_noise_sweep(raw_dir="results/c2/2026-11-10_153012/5.9_noise_sweep")` |
| `measure_condition` | `(trials.csv の行のリスト, summary.csv の 1 行)`。1 つの条件で `SWEEP_TRIALS` 回測定し、密度行列からの値と平均・標準偏差をまとめる（`run_noise_sweep` の中で使う） | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `run_noise_sweep` | Tuple `(list of per-trial records, list of per-condition summaries)`. If `raw_dir` is given, the raw data and `trials.csv` and `summary.csv` are saved in that folder | `records, summary = run_noise_sweep(raw_dir="results/c2/2026-11-10_153012/5.9_noise_sweep")` |
| `measure_condition` | `(rows for trials.csv, one row for summary.csv)`. Measures `SWEEP_TRIALS` times under one condition and collects the density-matrix values, means, and standard deviations (used inside `run_noise_sweep`) | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>run_noise_sweep</code></td><td><code>raw_dir</code></td><td>条件・試行別のビット列と JSON と、結果の CSV（<code>trials.csv</code>, <code>summary.csv</code>）の保存フォルダ。文字列または Path。<code>None</code> なら raw data も CSV も保存しない。空でないフォルダを指定すると測定前にエラーになるので、再実行時は未使用の保存先を指定</td><td><code>None</code></td></tr>
<tr><td rowspan="5"><code>measure_condition</code></td><td><code>component</code>, <code>strength</code></td><td>ノイズの成分の名前と強さ</td><td>必須</td></tr>
<tr><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>nm</code></td><td><code>build_sweep_noise(component, strength)</code> の NoiseModel</td><td>必須</td></tr>
<tr><td><code>raw_dir</code></td><td>raw data の保存先（<code>None</code> なら保存しない）</td><td>必須</td></tr>
<tr><td><code>first_seed</code></td><td>1 回目の試行のシード。試行ごとに 1 ずつ増やす（<code>None</code> なら毎回違う結果）。試行の回数と測定回数は <code>SWEEP_TRIALS</code>・<code>SWEEP_SHOTS</code> で決まる</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>run_noise_sweep</code></td><td><code>raw_dir</code></td><td>Folder in which to save the bit strings and JSON for each condition and trial, and the CSV files of the results (<code>trials.csv</code>, <code>summary.csv</code>), as a string or Path. <code>None</code> means neither the raw data nor the CSV files are saved. A non-empty folder raises an error before measuring, so specify an unused destination when re-running</td><td><code>None</code></td></tr>
<tr><td rowspan="5"><code>measure_condition</code></td><td><code>component</code>, <code>strength</code></td><td>Name and strength of the noise component</td><td>Required</td></tr>
<tr><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>nm</code></td><td>NoiseModel from <code>build_sweep_noise(component, strength)</code></td><td>Required</td></tr>
<tr><td><code>raw_dir</code></td><td>Destination of the raw data (<code>None</code>: not saved)</td><td>Required</td></tr>
<tr><td><code>first_seed</code></td><td>Seed of the first trial, increased by 1 for each trial (<code>None</code>: different results each time). The number of trials and measurements are set by <code>SWEEP_TRIALS</code> and <code>SWEEP_SHOTS</code></td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- 引数以外の実験条件は直前の設定セルで変更します。 -->

Experimental conditions other than the arguments are changed in the preceding settings cell.

<!--
| 設定変数 | 入力する値 |
|---|---|
| `SWEEP_VALUES` | 成分名（5.2 の `NOISE_COMPONENTS` の名前）をキー、試す強さのリストを値とする辞書 |
| `SWEEP_GATES` | 比較するゲートのリスト（例：`["H", "Rx", "Ry"]`） |
| `SWEEP_SHOTS` | 1 試行あたりの測定回数（shot 数）。1 以上の整数 |
| `SWEEP_TRIALS` | 条件ごとの試行数。標準偏差の計算のため 2 以上の整数 |
| `SWEEP_SEED` | 非負整数または `None` |
| `SWEEP_RAW_DIR` | 次のコードセル末尾の呼び出しに渡す保存先。1. で `SAVE_SWEEP = True` なら `RUN_DIR/5.9_noise_sweep`、`False` なら `None` に自動で決まる |
-->

| Setting variable | Value to set |
|---|---|
| `SWEEP_VALUES` | Dictionary with component names (names in `NOISE_COMPONENTS` in 5.2) as keys and lists of strengths to try as values |
| `SWEEP_GATES` | List of gates to compare (e.g., `["H", "Rx", "Ry"]`) |
| `SWEEP_SHOTS` | Number of measurements (shots) per trial. An integer of at least 1 |
| `SWEEP_TRIALS` | Number of trials per condition. An integer of at least 2, for computing the standard deviation |
| `SWEEP_SEED` | Non-negative integer or `None` |
| `SWEEP_RAW_DIR` | Destination passed to the call at the end of the next code cell. Set automatically: `RUN_DIR/5.9_noise_sweep` if `SAVE_SWEEP = True` in 1., otherwise `None` |


In [ ]:
SWEEP_METRICS = ("P1", "bias", "shannon", "min_entropy", "joint_min_entropy_per_bit", "joint_max_dev")


def measure_condition(component, strength, gate, nm, raw_dir, first_seed):
    # SWEEP_TRIALS measurements under one condition -> (rows for trials.csv, one row for summary.csv)
    rows = []
    for trial in range(1, SWEEP_TRIALS + 1):
        name = f"c2_{gate}_{component}-{strength:g}_trial{trial:03d}.txt"
        raw_path = None if raw_dir is None else Path(raw_dir) / component / name
        seed = offset_seed(first_seed, trial - 1)
        r = run_c2(gate, SWEEP_SHOTS, nm, seed, raw_path, method="density_matrix", optimization_level=0,
                   metadata={"component": component, "strength": strength, "trial": trial})
        rows.append(dict(component=component, strength=strength, gate=gate, trial=trial, seed=seed, **{k: r[k] for k in ("shots", "n_bits") + SWEEP_METRICS}))
    exact = sweep_exact(gate, component, strength, nm)
    summary = dict(component=component, strength=strength, gate=gate, **{"exact_" + k: v for k, v in exact.items()})
    for metric in SWEEP_METRICS:
        values = [row[metric] for row in rows]
        summary[metric + "_mean"] = statistics.mean(values)
        summary[metric + "_std"] = statistics.stdev(values)
    return rows, summary


def run_noise_sweep(raw_dir=None):
    if SWEEP_SHOTS < 1 or SWEEP_TRIALS < 2:
        raise ValueError("shots must be at least 1, and the number of trials must be at least 2 to compute the standard deviation")
    if raw_dir is not None and Path(raw_dir).exists() and any(Path(raw_dir).iterdir()):
        raise FileExistsError(f"Destination is not empty: {raw_dir}. Run the Setup cell (1.) again to save in a new folder.")
    conditions = len(SWEEP_GATES) * sum(map(len, SWEEP_VALUES.values()))
    print(f"{conditions} conditions × {SWEEP_TRIALS} trials × {SWEEP_SHOTS:,} shots"
          f" = {conditions * SWEEP_TRIALS * SWEEP_SHOTS:,} shots")
    records, summary = [], []
    for component, strengths in SWEEP_VALUES.items():
        for strength in strengths:
            nm = build_sweep_noise(component, strength)
            for gate in SWEEP_GATES:
                first_seed = offset_seed(SWEEP_SEED, len(records))
                rows, item = measure_condition(component, strength, gate, nm, raw_dir, first_seed)
                records += rows
                summary.append(item)
        print(f"Done: {component}")
    if raw_dir is not None:
        # The CSV files are saved only together with the raw data, after all conditions have been measured
        for name, data in [("trials.csv", records), ("summary.csv", summary)]:
            print("Saved:", write_csv(Path(raw_dir) / name, data))
    return records, summary


# Saved in RUN_DIR/5.9_noise_sweep only when SAVE_SWEEP = True in 1. Setup (None: not saved)
SWEEP_RAW_DIR = RUN_DIR / "5.9_noise_sweep" if SAVE_SWEEP else None
sweep_records, sweep_summary = run_noise_sweep(raw_dir=SWEEP_RAW_DIR)

In [ ]:
# Points and error bars: trial mean ± between-trial standard deviation (clipped to the range the metric can take)
# Solid lines: values computed from the density matrix
PLOT_ROWS = [("P1", "P(1) (flattened)", (0, 1)),
             ("bias", "Bias (flattened)", (0, 0.5)),
             ("joint_min_entropy_per_bit", "Joint min-entropy / 5", (0, 1))]


def plot_gate(ax_col, component, gate, color):
    # One gate in one column: sampled points with error bars and the exact line, in each row
    points = sorted((r for r in sweep_summary if r["component"] == component and r["gate"] == gate),
                    key=lambda r: r["strength"])
    xs = [r["strength"] for r in points]
    for ax, (metric, ylabel, (low, high)) in zip(ax_col, PLOT_ROWS):
        means = [r[metric + "_mean"] for r in points]
        stds = [r[metric + "_std"] for r in points]
        yerr = [[min(s, m - low) for m, s in zip(means, stds)],
                [min(s, high - m) for m, s in zip(means, stds)]]
        ax.errorbar(xs, means, yerr=yerr, fmt="o", capsize=3, color=color, label=gate + " sampled")
        ax.plot(xs, [r["exact_" + metric] for r in points], color=color, label=gate + " exact")
        ax.set(xlabel=NOISE_COMPONENTS[component]["label"], ylabel=ylabel)
        ax.grid(alpha=0.3)


joint_ref = expected_joint_min_entropy(SWEEP_SHOTS) / N_QUBITS
n_cols = len(SWEEP_VALUES)   # one column per noise component
fig, axes = plt.subplots(len(PLOT_ROWS), n_cols, figsize=(5 * n_cols, 11), squeeze=False)
for col, component in enumerate(SWEEP_VALUES):
    ax_col = axes[:, col]
    for gate_index, gate in enumerate(SWEEP_GATES):
        plot_gate(ax_col, component, gate, f"C{gate_index}")
    ax_col[0].axhline(0.5, color="gray", linestyle=":")
    fair_line = ax_col[1].axhline(expected_bias(N_QUBITS * SWEEP_SHOTS), color="gray", linestyle=":",
                                  label="Fair-bit expected sampled bias")
    joint_line = ax_col[2].axhline(joint_ref, color="gray", linestyle=":", label="Uniform expected sampled value")
    ax_col[0].legend(fontsize=8)
    ax_col[1].legend(handles=[fair_line], fontsize=8)
    ax_col[2].legend(handles=[joint_line], fontsize=8)
    if NOISE_COMPONENTS[component].get("log_scale"):
        # Multiplier 0 is included, so use a linear scale for 0–1 and a logarithmic scale above 1
        for ax in ax_col:
            ax.set_xscale("symlog", linthresh=1)
            ax.set_xlim(-0.3, max(SWEEP_VALUES[component]) * 1.5)
fig.tight_layout()
plt.show()

lines = ["| noise | strength | gate | P(1) mean ± SD | exact P(1) | bias mean | joint min-entropy / 5 mean | exact joint min-entropy / 5 |",
         "|---|---|---|---|---|---|---|---|"]
for r in sweep_summary:
    lines.append(f"| {r['component']} | {r['strength']:g} | {r['gate']} | "
                 f"{r['P1_mean']:.6f} ± {r['P1_std']:.6f} | {r['exact_P1']:.6f} | "
                 f"{r['bias_mean']:.6f} | {r['joint_min_entropy_per_bit_mean']:.5f} | "
                 f"{r['exact_joint_min_entropy_per_bit']:.5f} |")
display(Markdown("\n".join(lines)))

<!-- #### 結果の読み方と保存 -->

#### How to read the results and save them

<!-- **グラフは 3 行 × 3 列です。** 列と横軸の意味は C1 のノートブックの 5.9 と同じです。 -->

**The figure has 3 rows × 3 columns.** The columns and horizontal axes have the same meaning as in 5.9 of the C1 notebook.

<!--
| 段（縦軸） | 値の意味 |
|---|---|
| 上段：P(1) (flattened) | 連結したビット列で 1 が出る確率。偏りがないときは 0.5 |
| 中段：Bias (flattened) | 連結したビット列の P(1) と 0.5 の差の絶対値。0 に近いほど偏りが小さい |
| 下段：Joint min-entropy / 5 | 5 ビットの同時分布の min-entropy を 5 で割った値。1 に近いほど同時分布が一様に近い |
-->

| Row (vertical axis) | Meaning of the value |
|---|---|
| Top: P(1) (flattened) | Probability of getting 1 in the concatenated bit string. 0.5 when there is no bias |
| Middle: Bias (flattened) | Absolute difference between P(1) of the concatenated bit string and 0.5. The closer to 0, the smaller the bias |
| Bottom: Joint min-entropy / 5 | Min-entropy of the 5-bit joint distribution divided by 5. The closer to 1, the closer the joint distribution is to uniform |

<!--
| 列 | 横軸の英語表記 | 横軸の意味 |
|---|---|---|
| 左：脱分極 | Depolarizing parameter lambda | 脱分極パラメータ λ。0 は脱分極なし |
| 中央：熱緩和 | Thermal gate-time multiplier | 熱緩和の計算に使うゲート時間の倍率。0 は熱緩和なし、1 は元の 20 ns。0〜1 は線形、1 以上は対数の目盛（symlog） |
| 右：読み出し誤差 | Symmetric readout error epsilon | 0→1 と 1→0 を読み間違える確率 $\varepsilon$ |
-->

| Column | Horizontal-axis label | Meaning of the horizontal axis |
|---|---|---|
| Left: depolarizing | Depolarizing parameter lambda | Depolarizing parameter λ. 0 means no depolarizing |
| Center: thermal relaxation | Thermal gate-time multiplier | Multiplier on the gate time used in the thermal relaxation calculation. 0 means no thermal relaxation, and 1 is the original 20 ns. Linear from 0 to 1 and logarithmic above 1 (symlog) |
| Right: readout error | Symmetric readout error epsilon | Probability $\varepsilon$ of misreading 0→1 and 1→0 |

<!-- **点・実線・エラーバーの見方** -->

**How to read the points, solid lines, and error bars**

<!--
- 色は H・Rx・Ry の違いを表します（凡例や表の H は $H_{\mathrm{Gate}}$ です）。
- 点（`sampled`）は有限 shots の測定を繰り返して得た指標の平均、上下の棒は試行間の標準偏差です（信頼区間ではありません）。指標が取り得る範囲で切って表示しているので、切った側の棒は標準偏差より短くなります。
- 実線（`exact`）は、設定したノイズモデルから密度行列を使って求めた値です。有限 shots による統計的な揺らぎを含みません。
- 中段の灰色の点線は公平なビットを $5 \times$ `SWEEP_SHOTS` 個測定した場合の bias の目安、下段の灰色の点線は一様分布から `SWEEP_SHOTS` 回抽出した場合の同時分布の min-entropy / 5 の目安です。
-->

- Colors distinguish H, Rx, and Ry (H in the legend and tables is $H_{\mathrm{Gate}}$).
- The points (`sampled`) are the average of the metric obtained by repeating measurements with a finite number of shots, and the bars above and below are the standard deviation between trials (they are not confidence intervals). Because they are clipped to the range the metric can take, a clipped bar is shorter than the standard deviation on that side.
- The solid lines (`exact`) are values obtained from the configured noise model using density matrices. They contain no statistical fluctuation from finite shots.
- The gray dotted line in the middle row is the reference bias when $5 \times$ `SWEEP_SHOTS` fair bits are measured, and the gray dotted line in the bottom row is the reference min-entropy / 5 of the joint distribution when sampling `SWEEP_SHOTS` times from the uniform distribution.

<!-- **結果を読む順序** -->

**Order in which to read the results**

<!--
1. 上段・中段は、C1 のノートブックの 5.9 と同じ形になるはずです。C2 回路では各量子ビットに同じノイズが独立に加わるので、連結したビット列の $P(1)$ は 1 個の量子ビットの $P(1)$ と同じだからです。熱緩和を大きくすると $P(1)$ が下がり、$H_{\mathrm{Gate}}$ の変化が Rx・Ry より小さくなる理由も C1 と同じです。
2. 下段の実線は、ノイズがない（強さ 0）ときに 1 になります。一方、点は灰色の点線（約 0.993）の近くにあり、1 より小さくなっています。これは説明パート C で述べた「32 個の頻度の最大値は $1/32$ より大きくなる」という揺らぎによるもので、回路やノイズのせいではありません。
3. 熱緩和の倍率を大きくすると、各量子ビットが 0 に偏るので、32 通りのうち `'00000'` が最も出やすくなり、同時分布の min-entropy は下がります。下段の実線の下がり方は、上段の $P(1)$ の偏りから計算できる値 $-\log_2 \max(P(0), P(1))$ と同じです（量子ビットが独立だからです）。
4. 左列と右列では、脱分極や対称な読み出し誤差を大きくしても、$P(1)$ はほぼ 1/2、同時分布もほぼ一様のままです。水平な線は、ノイズが測定分布の偏りに現れていないことを示し、ノイズがないことを意味しません。
-->

1. The top and middle rows should have the same shape as in 5.9 of the C1 notebook. This is because, in the C2 circuit, the same noise is applied independently to each qubit, so $P(1)$ of the concatenated bit string equals $P(1)$ of a single qubit. The reasons why $P(1)$ decreases as thermal relaxation grows, and why the change for $H_{\mathrm{Gate}}$ is smaller than for Rx and Ry, are also the same as in C1.
2. The solid line in the bottom row equals 1 when there is no noise (strength 0). The points, on the other hand, lie near the gray dotted line (about 0.993), below 1. This is due to the fluctuation described in Explanation Part C, "the maximum of the 32 frequencies is larger than $1/32$," and is not caused by the circuit or the noise.
3. As the thermal relaxation multiplier increases, each qubit is biased toward 0, so `'00000'` becomes the most likely of the 32 outcomes and the min-entropy of the joint distribution decreases. The decline of the solid line in the bottom row matches the value $-\log_2 \max(P(0), P(1))$ that can be computed from the bias of $P(1)$ in the top row (because the qubits are independent).
4. In the left and right columns, even when the depolarizing or symmetric readout error is increased, $P(1)$ remains nearly 1/2 and the joint distribution remains nearly uniform. A horizontal line indicates that the noise does not show up as a bias in the measurement distribution; it does not mean that there is no noise.

<!-- **このモデルの限界**：このモデルでは量子ビットごとの誤差がすべて同じで、量子ビット間の干渉もありません。論文の Odra 5 で見られた量子ビットごとのばらつきや、全体に 1 が少ない傾向を調べるには、量子ビットごとに異なる非対称な読み出し誤差（`noise_model.add_readout_error(err, [q])` で量子ビット `q` だけに加える）などを入れる必要があります。 -->

**Limitations of this model**: In this model, all per-qubit errors are identical and there is no interference between qubits. To study the qubit-to-qubit variation and the overall tendency toward fewer 1s seen on Odra 5 in the paper, one would need to include, for example, asymmetric readout errors that differ per qubit (`noise_model.add_readout_error(err, [q])` adds one to qubit `q` only).

<!-- ここでの min-entropy は測定分布からの値であり、量子的な乱数の安全性を保証する値ではありません。 -->

The min-entropy here is a value derived from the measurement distribution and does not guarantee the security of the quantum random numbers.

<!-- **結果と raw data の保存** -->

**Saving the results and the raw data**

<!-- 1. で `SAVE_SWEEP = True` にした場合、`run_noise_sweep` は測定時に各条件・試行の連結したビット列（`.txt`）と条件・集計結果（`.txt.json`）を `RUN_DIR/5.9_noise_sweep/<ノイズの種類>/` に保存し、すべての条件の測定が終わったあとに、`5.9_noise_sweep/` に次の CSV を保存します。 -->

If `SAVE_SWEEP = True` in 1., `run_noise_sweep` saves the concatenated bit string (`.txt`) and the conditions and aggregated results (`.txt.json`) for each condition and trial in `RUN_DIR/5.9_noise_sweep/<noise type>/` during measurement, and after all conditions have been measured, it saves the following CSV files in `5.9_noise_sweep/`.

<!--
- `trials.csv`：試行ごとの全結果。
- `summary.csv`：条件ごとの平均・標準偏差・密度行列による値。
-->

- `trials.csv`: all results for each trial.
- `summary.csv`: per-condition means, standard deviations, and density-matrix values.

<!--
- `SAVE_SWEEP = False`（既定）の場合は、raw data も CSV も保存しません。
- CSV は、同じ実行の raw data と同じフォルダにだけ保存します。CSV だけが新しい結果に書き換わって、raw data と対応しなくなることはありません。
- 同じ `RUN_DIR` のまま測定のセルをもう一度実行すると、保存先が空でないので、測定を始める前にエラーになります。もう一度保存するときは、1. のセルを実行し直してください。
- 測定が途中で止まった場合、そのフォルダには CSV が作られません。CSV がないフォルダは、途中までの raw data です。
- CSV の集計結果から測定順のビット列は復元できないため、raw data が必要なら測定前に `SAVE_SWEEP = True` にしてください。
-->

- With `SAVE_SWEEP = False` (the default), neither the raw data nor the CSV files are saved.
- The CSV files are saved only in the same folder as the raw data of the same run. The CSV files are never replaced by newer results that no longer match the raw data.
- If you run the measurement cell again with the same `RUN_DIR`, the destination is not empty, so an error is raised before any measurement. To save again, rerun the cell in 1.
- If the measurement stops partway, no CSV files are created in that folder. A folder without CSV files holds the raw data of an incomplete run.
- The bit strings in measurement order cannot be recovered from the aggregated results in the CSV, so if you need raw data, set `SAVE_SWEEP = True` before measuring.

<!-- 下のセルは、保存した CSV の場所を表示します。 -->

The cell below shows where the CSV files were saved.

In [ ]:
# The CSV files are saved by run_noise_sweep in the cell above, in SWEEP_RAW_DIR together with the raw data.
if SWEEP_RAW_DIR is None:
    print("SAVE_SWEEP = False, so neither the raw data nor the CSV files were saved.")
else:
    for filename in ("trials.csv", "summary.csv"):
        path = Path(SWEEP_RAW_DIR) / filename
        print(("Saved: " if path.is_file() else "Not saved: ") + str(path))